# Value and Distance Result Analysis


This notebook presents the results for the paper On the Value of Non-i.i.d. Tabular IoT Data for Federated Learning. The notebook should be possible to run from beginning to end. In case of an error, please leave an issue in the GitHub. It will be handled as soon as possible.

Imports, as well as function definitions.

In [56]:
import os
import sys

# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["KERAS_BACKEND"] = "torch"
module_path = os.path.abspath(os.path.join('..' + os.sep + '..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [57]:
import pandas as pd
import scipy

In [58]:
CONST_ROUTE_MAIN_DIR = ".." + os.sep + ".." + os.sep

In [59]:
def get_data_from_route(dataset_name, type_of_partition, additional_parameter, random_seed):
    if type_of_partition == "manual":
        path_to_train_datasets = CONST_ROUTE_MAIN_DIR + "data" + os.sep + "partitioned_training_data" + os.sep + type_of_partition + os.sep + additional_parameter + os.sep + str(random_seed)
    else:
        path_to_train_datasets = CONST_ROUTE_MAIN_DIR + "data" + os.sep + "partitioned_training_data" + os.sep + type_of_partition + os.sep + "dataset_" + dataset_name + os.sep + "alpha_" + additional_parameter + os.sep + str(random_seed)

    return path_to_train_datasets

In [60]:
def get_results_from_route(aggregation_method, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, exclude_maverick=False):
    # path_to_result_dataframes = CONST_ROUTE_MAIN_DIR + "results" + os.sep + "FedAvg" + os.sep + dataset_name + os.sep + type_of_partition + os.sep + additional_parameter + os.sep + "mlp"
    if type_of_partition == "dirichlet":
        path_to_result_dataframes = CONST_ROUTE_MAIN_DIR + "results" + os.sep + "dataframes" + os.sep + aggregation_method + os.sep + dataset_name + os.sep + type_of_partition + os.sep + "alpha_" + str(additional_parameter) + os.sep + "mlp" + os.sep + str(random_seed) + os.sep + str(random_seed)
    else:
        if is_maverick_scenario:
            if exclude_maverick:
                path_to_result_dataframes = CONST_ROUTE_MAIN_DIR + "results" + os.sep + "dataframes" + os.sep + aggregation_method + os.sep + dataset_name + os.sep + type_of_partition + os.sep + additional_parameter+"_excl_maverick" + os.sep + "mlp" + os.sep + str(random_seed) + os.sep + str(random_seed)
            else:
                path_to_result_dataframes = CONST_ROUTE_MAIN_DIR + "results" + os.sep + "dataframes" + os.sep + aggregation_method + os.sep + dataset_name + os.sep + type_of_partition + os.sep + additional_parameter+"_incl_maverick" + os.sep + "mlp" + os.sep + str(random_seed) + os.sep + str(random_seed)
        else:
            path_to_result_dataframes = CONST_ROUTE_MAIN_DIR + "results" + os.sep + "dataframes" + os.sep + aggregation_method + os.sep + dataset_name + os.sep + type_of_partition + os.sep + additional_parameter + os.sep + "mlp" + os.sep + str(random_seed) + os.sep + str(random_seed)
    return path_to_result_dataframes

In [61]:
def get_distances_from_route(dataset_name, type_of_partition, additional_parameter, random_seed):
    path_to_result_dataframes = CONST_ROUTE_MAIN_DIR + "results" + os.sep + "distances_values" + os.sep + dataset_name + os.sep + type_of_partition + os.sep + str(additional_parameter) + os.sep + str(random_seed)
    return path_to_result_dataframes

In [62]:
from itertools import combinations_with_replacement

def number_of_clients_and_all_combinations(path_to_train_datasets):
    num_clients = int(len(os.listdir(path_to_train_datasets)) / 4)
    client_numbers_original_order = list(range(num_clients))
    client_numbers_reverse_order = list(range(num_clients - 1, -1, -1))

    all_combinations = list(combinations_with_replacement(client_numbers_original_order, 2)) + list(combinations_with_replacement(client_numbers_reverse_order, 2))
    all_combinations = sorted(list(set(all_combinations)))
    print(all_combinations)
    return num_clients, all_combinations

In [63]:
def downcast_types(dataframe):
    for column in dataframe.select_dtypes("int"):
        dataframe[column] = dataframe[column].astype("int16")

    for column in dataframe.select_dtypes("float"):
        dataframe[column] = dataframe[column].astype("float32")

    return dataframe

# Distances measuring functions

In [64]:
# use_cuda = torch.cuda.is_available()
# dtype = torch.cuda.FloatTensor if use_cuda else torch.FloatTensor
# print(use_cuda)

!Comment about the prince library.

In [65]:
import itertools

def generate_experiments() -> list[dict]:
    # Valid (strategy, model) pairs — matched by shared type key (nn/dt)

    # strategies = {"nn": ["FedAvg"], "dt": ["FedXGB"]}
    # models = {"nn": ["mlp"], "dt": ["xgboost"]}
    strategies = {"nn": ["FedAvg", "FedProx", "Scaffold"]}
    # strategies = {"nn": ["FedAvg", "FedProx"]}
    # strategies = {"nn": ["FedAvg"]}
    # strategies = {"nn": ["FedProx"]}
    models = {"nn": ["mlp"]}
    datasets = {
        "classification": [
            # "wine",
            "har",
        #     # "adult",
        #     # "covertype",
        #     # "heart",
        #     # "binary_heart"
            "edge-iot-coreset"
        ],
        "regression": [
            "wids_dataset"
        ]
    }

    partition_types = [
        "dirichlet",
        # "feature_skew",
        "manual"
    ]

    hyperparameter_search_rounds = 30
    rounds = 50

    metrics = {"classification": ["CrossEntropyLoss", "Accuracy", "F1Score", "MCC", "F1ScoreMacro", "F1ScoreMicro"],
               "regression": ["MSE", "RMSE", "MAE"]}

    DEFAULT_DATA_SPLIT = [20, 20, 20, 20, 20]
    data_split = {
        "har": [16, 16, 16, 16, 16, 16],
    }

    alpha = [
        0.1,
        10,
        1000,
    ]
    n_clients = [5]
    run_number_list = [number for number in range(1, 6)]
    # run_number_list = [1]

    # Manual partitions come in two shapes:
    #  - a plain list of names (no maverick semantics)
    #  - a dict {name: (is_maverick, maverick_client_number, eval_type, special_class)}
    #    for scenarios where one client is a "maverick" holding unique data. I separate between Mavericks of class
    #    and feature by string.
    manual_partition_names = {
        "har": {
            "HAR_1_Maverick_Laying": (True, 0, "Class", 0, None),
            "HAR_1_Maverick_WalkingUpstairs": (True, 5, "Class", 5, None),
            "HAR_1_Maverick_Laying_Balanced": (True, 0, "Class", 0, None),
            "HAR_1_Maverick_WalkingUpstairs_Balanced": (True, 5, "Class", 5, None),
            # "HAR_1_Maverick_1_MissingOneLabel": (True, 5, "Class", 5, None),
            # "HAR_1_Maverick_1_MissingTwoLabels": (True, 5, "Class", 5, None),
        },
        "edge-iot-coreset": {
            "edgeiot_coreset_1_Maverick_Least_Class": (True, 4, "Class", 6, None),
            "edgeiot_coreset_1_Maverick_Only_Normal": (True, 4, "Class", 7, None),
            "edgeiot_coreset_1_Maverick_sql_injection": (True, 4, "Class", 11, None),
            "edgeiot_coreset_1_Maverick_ddos_udp": (True, 4, "Class", 4, None),
        },
        "wids_dataset": {
            "wids_energy_non_iid_by_label": (False, None, None, None, None),
            "wids_energy_iid_sampling": (False, None, None, None, None),
            "wids_energy_feature_skew_building_type": (False, None, None, None, None),
            "wids_energy_feature_skew_state_factor": (False, None, None, None, None),
            # "wids_energy_feature_skew_facility_type": (False, None, None, None, None),
            "wids_energy_maverick_facility_type_grocery_store": (
                True, 5, "Feature", ["facility_type"], "Grocery_store_or_food_market"
            ),
            "wids_energy_maverick_facility_type_uncategorized_multifamily": (
                True, 5, "Feature", ["facility_type"], "Multifamily_Uncategorized"
            ),
        }
    }

    strategy_model_pairs = [
        (strategy, model)
        for type_key in set(strategies) & set(models)
        for strategy in strategies[type_key]
        for model in models[type_key]
    ]

    dataset_metrics_pairs = [
        (dataset, metrics[type_key], type_key)
        for type_key in set(datasets) & set(metrics)
        for dataset in datasets[type_key]
    ]

    experiments = []

    for (strategy, model), (dataset, selected_metrics, task_type), number in itertools.product(
            strategy_model_pairs, dataset_metrics_pairs, run_number_list
    ):
        base = {
            "strategy": strategy,
            "model": model,
            "dataset": dataset,
            "metrics": selected_metrics,
            "run_number": number
        }

        # Regression tasks are restricted to manual partitions only.
        allowed_partitions = ["manual"] if task_type == "regression" else partition_types

        for partition in allowed_partitions:
            entry = {**base, "partition": partition}

            if partition == "dirichlet":
                split = data_split.get(dataset, DEFAULT_DATA_SPLIT)
                for a in alpha:
                    experiments.append({**entry, "alpha": a, "data_split": split})

            elif partition == "feature_skew":
                for nc in n_clients:
                    experiments.append({**entry, "n_clients": nc})

            elif partition == "manual":
                partitions = manual_partition_names.get(dataset, [])

                if isinstance(partitions, dict):
                    # Maverick-style entries — generate both a "with maverick"
                    # and a "without maverick" training run per scenario.
                    for name, (is_maverick, maverick_client_number, eval_type,
                               special_class_or_columns_for_subpopulation,
                               maverick_category_value) in partitions.items():
                        if is_maverick:
                            for exclude_maverick in (True, False):
                                experiments.append({
                                    **entry,
                                    "name": name,
                                    "is_maverick_scenario": True,
                                    "maverick_client_number": maverick_client_number,
                                    "maverick_eval_type": eval_type,
                                    "maverick_special_class_or_columns_for_subpopulation": special_class_or_columns_for_subpopulation,
                                    "maverick_category_value": maverick_category_value,
                                    "exclude_maverick": exclude_maverick,
                                })
                        else:
                            experiments.append({**entry, "name": name})
                else:
                    for name in partitions:
                        experiments.append({**entry, "name": name})

    return experiments

training_configurations_for_distances = generate_experiments()
print(training_configurations_for_distances)

[{'strategy': 'FedAvg', 'model': 'mlp', 'dataset': 'wids_dataset', 'metrics': ['MSE', 'RMSE', 'MAE'], 'run_number': 1, 'partition': 'manual', 'name': 'wids_energy_non_iid_by_label'}, {'strategy': 'FedAvg', 'model': 'mlp', 'dataset': 'wids_dataset', 'metrics': ['MSE', 'RMSE', 'MAE'], 'run_number': 1, 'partition': 'manual', 'name': 'wids_energy_iid_sampling'}, {'strategy': 'FedAvg', 'model': 'mlp', 'dataset': 'wids_dataset', 'metrics': ['MSE', 'RMSE', 'MAE'], 'run_number': 1, 'partition': 'manual', 'name': 'wids_energy_feature_skew_building_type'}, {'strategy': 'FedAvg', 'model': 'mlp', 'dataset': 'wids_dataset', 'metrics': ['MSE', 'RMSE', 'MAE'], 'run_number': 1, 'partition': 'manual', 'name': 'wids_energy_feature_skew_state_factor'}, {'strategy': 'FedAvg', 'model': 'mlp', 'dataset': 'wids_dataset', 'metrics': ['MSE', 'RMSE', 'MAE'], 'run_number': 1, 'partition': 'manual', 'name': 'wids_energy_maverick_facility_type_grocery_store', 'is_maverick_scenario': True, 'maverick_client_number':

In [66]:
def get_results_of_training(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, exclude_maverick):
    results_dataframe = pd.Series()

    path_to_result_dataframes = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, exclude_maverick)

    # metrics = []
    # result = []

    results_dataframe.loc["Aggregation_Process"] = aggregation_process
    results_dataframe.loc["Dataset"] = dataset_name
    results_dataframe.loc["Type_of_partition"] = type_of_partition
    results_dataframe.loc["Dirichlet/NamePartition"] = additional_parameter
    results_dataframe.loc["Random_Seed"] = random_seed
    results_dataframe.loc["Exclude_Maverick"] = exclude_maverick

    for file in os.listdir(path_to_result_dataframes + os.sep + "Evaluation"):
        # spearman_rank_distance_sv = pd.DataFrame(columns=["Dataset", "TypePartition", "AdditionalParameter", "Metric", "Evaluator", "Distance", "SpearmanRank", "p-value"])
        metric_name = file.split("_")[1]

        if file != "Evaluation_F1Score" and "Subpopulations" not in file:
            evaluation_dataframe = pd.read_csv(path_to_result_dataframes + os.sep + "Evaluation" + os.sep + file)
            results_dataframe.loc[metric_name] = float(evaluation_dataframe.loc[evaluation_dataframe.index[-1], "Global"])
            results_dataframe.loc[metric_name + "_decentralized"] = float(evaluation_dataframe.loc[evaluation_dataframe.index[-1], "Aggregated"])

    return results_dataframe

In [67]:
results_dataframe = pd.DataFrame()

for configuration in training_configurations_for_distances:
    aggregation_process = configuration["strategy"]
    dataset_name = configuration["dataset"]
    type_of_partition = configuration["partition"]
    if type_of_partition == "dirichlet":
        additional_parameter = configuration["alpha"]
        is_maverick_scenario = False
        exclude_maverick = False
    elif type_of_partition == "manual":
        additional_parameter = configuration["name"]
        is_maverick_scenario = configuration.get("is_maverick_scenario", False)
        exclude_maverick = configuration.get("exclude_maverick", False)
    else:
        raise NotImplementedError
    # additional_parameter = configuration["Dirichlet/NamePartition"]
    random_seed = configuration["run_number"]

    results_dataframe = pd.concat([results_dataframe, get_results_of_training(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, exclude_maverick).to_frame().T], ignore_index=True)

In [68]:
results_dataframe = results_dataframe.convert_dtypes()
display(results_dataframe.dtypes)

Aggregation_Process               string[python]
Dataset                           string[python]
Type_of_partition                 string[python]
Dirichlet/NamePartition                   object
Random_Seed                                Int64
Exclude_Maverick                         boolean
RMSE                                     Float64
RMSE_decentralized                       Float64
MAE                                      Float64
MAE_decentralized                        Float64
MSE                                      Float64
MSE_decentralized                        Float64
F1ScoreMicro                             Float64
F1ScoreMicro_decentralized               Float64
F1ScoreMacro                             Float64
F1ScoreMacro_decentralized               Float64
Accuracy                                 Float64
Accuracy_decentralized                   Float64
CrossEntropyLoss                         Float64
CrossEntropyLoss_decentralized           Float64
MCC                 

In [69]:
display(
    results_dataframe[((results_dataframe["Exclude_Maverick"] == False) & (results_dataframe["Dataset"] != "wids_dataset"))].groupby(["Aggregation_Process", "Dataset", "Type_of_partition", "Dirichlet/NamePartition"]).mean(numeric_only=True)[
        [
        "CrossEntropyLoss",
        "Accuracy",
        "F1ScoreMacro",
        "F1ScoreMicro",
        "MCC"]
    ]
)

CrossEntropyLoss  \
Aggregation_Process Dataset          Type_of_partition Dirichlet/NamePartition                                      
FedAvg              edge-iot-coreset dirichlet         0.1                                               0.552129   
                                                       10                                                 0.48738   
                                                       1000                                               0.36447   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class             0.35725   
                                                       edgeiot_coreset_1_Maverick_Only_Normal            0.398221   
                                                       edgeiot_coreset_1_Maverick_ddos_udp                0.36117   
                                                       edgeiot_coreset_1_Maverick_sql_injection          0.534662   
                    har              dirichlet         0.1                                               0.376992   
                                                       10                                                0.148966   
                                                       1000                                              0.143723   
                                     manual            HAR_1_Maverick_Laying                             0.145344   
                                                       HAR_1_Maverick_Laying_Balanced                    0.214947   
                                                       HAR_1_Maverick_WalkingUpstairs                    0.195022   
                                                       HAR_1_Maverick_WalkingUpstairs_Balanced           0.176349   
FedProx             edge-iot-coreset dirichlet         0.1                                               0.807338   
                                                       10                                                0.529689   
                                                       1000                                              0.398199   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class            0.399876   
                                                       edgeiot_coreset_1_Maverick_Only_Normal            0.443741   
                                                       edgeiot_coreset_1_Maverick_ddos_udp               0.449731   
                                                       edgeiot_coreset_1_Maverick_sql_injection          0.481196   
                    har              dirichlet         0.1                                               0.515447   
                                                       10                                                0.163853   
                                                       1000                                              0.135528   
                                     manual            HAR_1_Maverick_Laying                             0.138549   
                                                       HAR_1_Maverick_Laying_Balanced                    0.220565   
                                                       HAR_1_Maverick_WalkingUpstairs                    0.224165   
                                                       HAR_1_Maverick_WalkingUpstairs_Balanced           0.168227   
Scaffold            edge-iot-coreset dirichlet         0.1                                               0.612381   
                                                       10                                                0.490146   
                                                       1000                                              0.364294   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class            0.359501   
                                                       edgeiot_coreset_1_Maverick_Only_Normal            0.404481   
  

In [70]:
display(
    results_dataframe[((results_dataframe["Exclude_Maverick"] == False) & (results_dataframe["Dataset"] != "wids_dataset"))].groupby(["Aggregation_Process", "Dataset", "Type_of_partition", "Dirichlet/NamePartition"]).std(numeric_only=True)[
        [
        "CrossEntropyLoss",
        "Accuracy",
        "F1ScoreMacro",
        "F1ScoreMicro",
        "MCC"]
    ]
)

CrossEntropyLoss  \
Aggregation_Process Dataset          Type_of_partition Dirichlet/NamePartition                                      
FedAvg              edge-iot-coreset dirichlet         0.1                                               0.034459   
                                                       10                                                0.040313   
                                                       1000                                              0.004542   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class            0.005096   
                                                       edgeiot_coreset_1_Maverick_Only_Normal            0.016696   
                                                       edgeiot_coreset_1_Maverick_ddos_udp               0.010266   
                                                       edgeiot_coreset_1_Maverick_sql_injection          0.055975   
                    har              dirichlet         0.1                                                0.05801   
                                                       10                                                0.019257   
                                                       1000                                              0.010953   
                                     manual            HAR_1_Maverick_Laying                             0.012504   
                                                       HAR_1_Maverick_Laying_Balanced                    0.023589   
                                                       HAR_1_Maverick_WalkingUpstairs                    0.014532   
                                                       HAR_1_Maverick_WalkingUpstairs_Balanced            0.02135   
FedProx             edge-iot-coreset dirichlet         0.1                                               0.178449   
                                                       10                                                0.073082   
                                                       1000                                              0.006744   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class            0.005743   
                                                       edgeiot_coreset_1_Maverick_Only_Normal            0.017258   
                                                       edgeiot_coreset_1_Maverick_ddos_udp               0.044822   
                                                       edgeiot_coreset_1_Maverick_sql_injection          0.007843   
                    har              dirichlet         0.1                                               0.281824   
                                                       10                                                0.042508   
                                                       1000                                              0.010802   
                                     manual            HAR_1_Maverick_Laying                              0.00898   
                                                       HAR_1_Maverick_Laying_Balanced                    0.025032   
                                                       HAR_1_Maverick_WalkingUpstairs                    0.094072   
                                                       HAR_1_Maverick_WalkingUpstairs_Balanced           0.009698   
Scaffold            edge-iot-coreset dirichlet         0.1                                               0.046608   
                                                       10                                                0.037816   
                                                       1000                                              0.006046   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class             0.00245   
                                                       edgeiot_coreset_1_Maverick_Only_Normal            0.018313   
  

In [71]:
display(
    results_dataframe[((results_dataframe["Exclude_Maverick"] == False) & (results_dataframe["Dataset"] == "wids_dataset"))].groupby(["Aggregation_Process", "Dataset", "Type_of_partition", "Dirichlet/NamePartition"]).mean(numeric_only=True)[
        ["MSE", "RMSE", "MAE"]
    ]
)

MSE  \
Aggregation_Process Dataset      Type_of_partition Dirichlet/NamePartition                                           
FedAvg              wids_dataset manual            wids_energy_feature_skew_building_type              1668.518865   
                                                   wids_energy_feature_skew_state_factor               1574.106704   
                                                   wids_energy_iid_sampling                            1920.268181   
                                                   wids_energy_maverick_facility_type_grocery_store    1539.479846   
                                                   wids_energy_maverick_facility_type_uncategorize...  1562.468754   
                                                   wids_energy_non_iid_by_label                        2552.561213   
FedProx             wids_dataset manual            wids_energy_feature_skew_building_type              1757.950107   
                                                   wids_energy_feature_skew_state_factor               1598.216582   
                                                   wids_energy_iid_sampling                            1565.248595   
                                                   wids_energy_maverick_facility_type_grocery_store    1573.246278   
                                                   wids_energy_maverick_facility_type_uncategorize...   1545.98539   
                                                   wids_energy_non_iid_by_label                        2900.720482   
Scaffold            wids_dataset manual            wids_energy_feature_skew_building_type              1664.127002   
                                                   wids_energy_feature_skew_state_factor                1957.31207   
                                                   wids_energy_iid_sampling                            1564.307506   
                                                   wids_energy_maverick_facility_type_grocery_store    1552.197829   
                                                   wids_energy_maverick_facility_type_uncategorize...  1547.898767   
                                                   wids_energy_non_iid_by_label                        2751.610797   

                                                                                                            RMSE  \
Aggregation_Process Dataset      Type_of_partition Dirichlet/NamePartition                                         
FedAvg              wids_dataset manual            wids_energy_feature_skew_building_type              40.841416   
                                                   wids_energy_feature_skew_state_factor               39.674133   
                                                   wids_energy_iid_sampling                            43.107878   
                                                   wids_energy_maverick_facility_type_grocery_store    39.235176   
                                                   wids_energy_maverick_facility_type_uncategorize...  39.527085   
                                                   wids_energy_non_iid_by_label                        50.487287   
FedProx             wids_dataset manual            wids_energy_feature_skew_building_type              41.924146   
                                                   wids_energy_feature_skew_state_factor               39.976288   
                                                   wids_energy_iid_sampling                            39.562437   
                                                   wids_energy_maverick_facility_type_grocery_store    39.662901   
                                                   wids_energy_maverick_facility_type_uncategorize...  39.318439   
                                                   wids_energy_non_iid_by_label                        53.811802   
Scaffold            wids_dataset manual            wids_energy_feature_skew_building_type              40.791268   
          

In [72]:
display(
    results_dataframe[((results_dataframe["Exclude_Maverick"] == False) & (results_dataframe["Dataset"] == "wids_dataset"))].groupby(["Aggregation_Process", "Dataset", "Type_of_partition", "Dirichlet/NamePartition"]).std(numeric_only=True)[
        ["MSE", "RMSE", "MAE"]
    ]
)

MSE  \
Aggregation_Process Dataset      Type_of_partition Dirichlet/NamePartition                                          
FedAvg              wids_dataset manual            wids_energy_feature_skew_building_type                65.15881   
                                                   wids_energy_feature_skew_state_factor                23.448864   
                                                   wids_energy_iid_sampling                            862.642313   
                                                   wids_energy_maverick_facility_type_grocery_store     25.008303   
                                                   wids_energy_maverick_facility_type_uncategorize...   24.652939   
                                                   wids_energy_non_iid_by_label                        217.351005   
FedProx             wids_dataset manual            wids_energy_feature_skew_building_type               52.693201   
                                                   wids_energy_feature_skew_state_factor                30.058658   
                                                   wids_energy_iid_sampling                             22.070178   
                                                   wids_energy_maverick_facility_type_grocery_store     28.085175   
                                                   wids_energy_maverick_facility_type_uncategorize...   18.813569   
                                                   wids_energy_non_iid_by_label                        274.032185   
Scaffold            wids_dataset manual            wids_energy_feature_skew_building_type               41.031167   
                                                   wids_energy_feature_skew_state_factor               843.123743   
                                                   wids_energy_iid_sampling                             12.937374   
                                                   wids_energy_maverick_facility_type_grocery_store      21.03733   
                                                   wids_energy_maverick_facility_type_uncategorize...    27.42311   
                                                   wids_energy_non_iid_by_label                        261.620948   

                                                                                                           RMSE  \
Aggregation_Process Dataset      Type_of_partition Dirichlet/NamePartition                                        
FedAvg              wids_dataset manual            wids_energy_feature_skew_building_type              0.788681   
                                                   wids_energy_feature_skew_state_factor               0.295599   
                                                   wids_energy_iid_sampling                            8.801919   
                                                   wids_energy_maverick_facility_type_grocery_store    0.317849   
                                                   wids_energy_maverick_facility_type_uncategorize...  0.312909   
                                                   wids_energy_non_iid_by_label                        2.119857   
FedProx             wids_dataset manual            wids_energy_feature_skew_building_type              0.628581   
                                                   wids_energy_feature_skew_state_factor               0.375785   
                                                   wids_energy_iid_sampling                            0.278813   
                                                   wids_energy_maverick_facility_type_grocery_store     0.35457   
                                                   wids_energy_maverick_facility_type_uncategorize...  0.239132   
                                                   wids_energy_non_iid_by_label                        2.502608   
Scaffold            wids_dataset manual            wids_energy_feature_skew_building_type               0.49933   
                                            

## Establishing value, harm and harmless

In [142]:
# We use CELoss, as it is the proper scoring measure.

def performance_drop(results_dataframe):
    results_dataframe["PerformanceDrop"] = pd.NA

    # "wids_energy_non_iid_by_label", "wids_energy_iid_sampling",
    strategies = ["FedAvg", "FedProx", "Scaffold"]
    manual_partition_names = {
        "har": ["HAR_1_Maverick_Laying", "HAR_1_Maverick_WalkingUpstairs", "HAR_1_Maverick_Laying_Balanced", "HAR_1_Maverick_WalkingUpstairs_Balanced"],
        "edge-iot-coreset": ["edgeiot_coreset_1_Maverick_Least_Class", "edgeiot_coreset_1_Maverick_Only_Normal", "edgeiot_coreset_1_Maverick_sql_injection", "edgeiot_coreset_1_Maverick_ddos_udp"],
        "wids_dataset": ["wids_energy_maverick_facility_type_grocery_store", "wids_energy_maverick_facility_type_uncategorized_multifamily"]
    }
    iid_values = {
        "har": ("dirichlet", 1000),
        "edge-iot-coreset": ("dirichlet", 1000),
        "wids_dataset": ("manual", "wids_energy_iid_sampling")
    }
    non_iid_values = {
        "har": ("dirichlet", 0.1),
        "edge-iot-coreset": ("dirichlet", 0.1),
        "wids_dataset": ("manual", "wids_energy_non_iid_by_label")
    }

    for strategy in strategies:
        for dataset, manual_partition_name_list in manual_partition_names.items():
            if dataset == "wids_dataset":
                for name in manual_partition_name_list:
                    results_dataframe.loc[(strategy, dataset, "manual", name), "PerformanceDrop"] = (
                                abs(0.5 * (results_dataframe.loc[(strategy, dataset, *iid_values[dataset]), "MSE"] - results_dataframe.loc[(strategy, dataset, "manual", name), "MSE"]))
                                > abs(0.5 * (results_dataframe.loc[(strategy, dataset, "manual", name), "MSE"] - results_dataframe.loc[(strategy, dataset, *non_iid_values[dataset]), "MSE"]))
                    )
            else:
                for name in manual_partition_name_list:
                    results_dataframe.loc[(strategy, dataset, "manual", name), "PerformanceDrop"] = (
                                abs(0.5 * (results_dataframe.loc[(strategy, dataset, *iid_values[dataset]), "CrossEntropyLoss"] - results_dataframe.loc[(strategy, dataset, "manual", name), "CrossEntropyLoss"]))
                                > abs(0.5 * (results_dataframe.loc[(strategy, dataset, "manual", name), "CrossEntropyLoss"] - results_dataframe.loc[(strategy, dataset, *non_iid_values[dataset]), "CrossEntropyLoss"]))
                    )
            # results_dataframe.loc[, "PerformanceDrop"] = True

    return results_dataframe


In [145]:
results_aggregated = results_dataframe[((results_dataframe["Exclude_Maverick"] == False))].groupby(["Aggregation_Process", "Dataset", "Type_of_partition", "Dirichlet/NamePartition"]).mean(numeric_only=True)[["CrossEntropyLoss",
    "Accuracy",
    "F1ScoreMacro",
    "F1ScoreMicro",
    "MCC",
    "MSE"
    ]
]
performance_dropped_results_aggregated = performance_drop(results_aggregated)

                                                                                                           CrossEntropyLoss  \
Aggregation_Process Dataset          Type_of_partition Dirichlet/NamePartition                                                
FedAvg              edge-iot-coreset dirichlet         0.1                                                         0.552129   
                                                       10                                                           0.48738   
                                                       1000                                                         0.36447   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class                       0.35725   
                                                       edgeiot_coreset_1_Maverick_Only_Normal                      0.398221   
                                                       edgeiot_coreset_1_Maverick_ddos_udp                     

In [146]:
performance_dropped_results_aggregated

CrossEntropyLoss  \
Aggregation_Process Dataset          Type_of_partition Dirichlet/NamePartition                                                
FedAvg              edge-iot-coreset dirichlet         0.1                                                         0.552129   
                                                       10                                                           0.48738   
                                                       1000                                                         0.36447   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class                       0.35725   
                                                       edgeiot_coreset_1_Maverick_Only_Normal                      0.398221   
                                                       edgeiot_coreset_1_Maverick_ddos_udp                          0.36117   
                                                       edgeiot_coreset_1_Maverick_sql_injection                    0.534662   
                    har              dirichlet         0.1                                                         0.376992   
                                                       10                                                          0.148966   
                                                       1000                                                        0.143723   
                                     manual            HAR_1_Maverick_Laying                                       0.145344   
                                                       HAR_1_Maverick_Laying_Balanced                              0.214947   
                                                       HAR_1_Maverick_WalkingUpstairs                              0.195022   
                                                       HAR_1_Maverick_WalkingUpstairs_Balanced                     0.176349   
                    wids_dataset     manual            wids_energy_feature_skew_building_type                          <NA>   
                                                       wids_energy_feature_skew_state_factor                           <NA>   
                                                       wids_energy_iid_sampling                                        <NA>   
                                                       wids_energy_maverick_facility_type_grocery_store                <NA>   
                                                       wids_energy_maverick_facility_type_uncategorize...              <NA>   
                                                       wids_energy_non_iid_by_label                                    <NA>   
FedProx             edge-iot-coreset dirichlet         0.1                                                         0.807338   
                                                       10                                                          0.529689   
                                                       1000                                                        0.398199   
                                     manual            edgeiot_coreset_1_Maverick_Least_Class                      0.399876   
                                                       edgeiot_coreset_1_Maverick_Only_Normal                      0.443741   
                                                       edgeiot_coreset_1_Maverick_ddos_udp                         0.449731   
                                                       edgeiot_coreset_1_Maverick_sql_injection                    0.481196   
                    har              dirichlet         0.1                                                         0.515447   
                                                       10                                                          0.163853   
                                                       1000                                                        0.135528   
                                     manual

In [76]:
# display(
#     results_dataframe[((results_dataframe["Exclude_Maverick"] == False) & (results_dataframe["Dataset"] != "wids_dataset"))].groupby(["Aggregation_Process", "Dataset", "Type_of_partition", "Dirichlet/NamePartition"]).mean(numeric_only=True)[
#         [
#         "CrossEntropyLoss",
#         "Accuracy",
#         "F1ScoreMacro",
#         "F1ScoreMicro",
#         "MCC"]
#     ]
# )

In [77]:
def f1_score_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick_number, maverick_label):
    value_distance_dataframe = pd.DataFrame()

    path_to_result_dataframes_with_maverick = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True, False)
    path_to_result_dataframes_without_maverick = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True, True)

    f1_score_with_maverick = pd.read_csv(path_to_result_dataframes_with_maverick + os.sep + "Evaluation" + os.sep + "Evaluation_F1Score", index_col=0, header=[0, 1])
    f1_score_without_maverick = pd.read_csv(path_to_result_dataframes_without_maverick + os.sep + "Evaluation" + os.sep + "Evaluation_F1Score", index_col=0, header=[0, 1])

    path_to_data = get_data_from_route(dataset_name, type_of_partition, additional_parameter, random_seed)

    labels_with_maverick = pd.DataFrame()
    labels_without_maverick = pd.DataFrame()

    number_of_clients = int(len(os.listdir(path_to_data)) / 4) - 1

    for client_number in range(number_of_clients):
        if client_number == maverick_number:
            labels_without_maverick = pd.concat([labels_without_maverick, pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_mav_data_excluded_y_test.csv", index_col=0).sum(axis=0)).T], ignore_index=True)
            labels_with_maverick = pd.concat([labels_with_maverick, pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_y_test.csv", index_col=0).sum(axis=0)).T], ignore_index=True)
        else:
            labels = pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_y_test.csv", index_col=0).sum(axis=0)).T
            labels_with_maverick = pd.concat([labels_with_maverick, labels], ignore_index=True)
            labels_without_maverick = pd.concat([labels_without_maverick, labels], ignore_index=True)

    labels_with_maverick_proportion = labels_with_maverick.sum(axis=0) / labels_with_maverick.to_numpy().sum()
    labels_without_maverick_proportion = labels_without_maverick.sum(axis=0) / labels_without_maverick.to_numpy().sum()

    f1_score_global_with_maverick = f1_score_with_maverick.loc[f1_score_with_maverick.index[-1], "Global"]
    f1_score_global_without_maverick = f1_score_without_maverick.loc[f1_score_without_maverick.index[-1], "Global"]

    # Aggregated statistics with raw values
    f1_score_difference_computed = f1_score_global_with_maverick - f1_score_global_without_maverick
    f1_score_difference_computed_without_mav_label: pd.DataFrame = f1_score_difference_computed.copy()
    f1_score_difference_computed_without_mav_label.iloc[maverick_label] = 0
    max_loss = f1_score_difference_computed_without_mav_label.min()
    avg_loss = f1_score_difference_computed_without_mav_label.mean()
    max_gain_beside_maverick = f1_score_difference_computed_without_mav_label.max()
    maverick_gain_label = f1_score_difference_computed.iloc[maverick_label]

    value_distance_dataframe["AggregationProcess"] = [aggregation_process]
    value_distance_dataframe["Dataset"] = [dataset_name]
    value_distance_dataframe["Partition"] = [additional_parameter]
    value_distance_dataframe["Metric"] = ["F1Score"]
    value_distance_dataframe["RandomSeed"] = [random_seed]
    value_distance_dataframe["MaverickOnlyLabelGain"] = [maverick_gain_label]
    value_distance_dataframe["MaverickMaxLoss"] = [max_loss]
    value_distance_dataframe["MaverickAvgLoss"] = [avg_loss]
    value_distance_dataframe["MaverickMaxGainBesidesOnlyLabel"] = [max_gain_beside_maverick]

    # Aggregated statistics with raw values
    f1_score_weighted_with_maverick = f1_score_global_with_maverick * labels_with_maverick_proportion
    f1_score_weighted_without_maverick = f1_score_global_without_maverick * labels_without_maverick_proportion
    f1_score_weighted_difference_computed = f1_score_weighted_with_maverick - f1_score_weighted_without_maverick
    f1_score_weighted_difference_computed_without_mav_label: pd.DataFrame = f1_score_weighted_difference_computed.copy()
    f1_score_weighted_difference_computed_without_mav_label.iloc[maverick_label] = 0
    max_loss = f1_score_weighted_difference_computed_without_mav_label.min()
    avg_loss = f1_score_weighted_difference_computed_without_mav_label.mean()
    max_gain_beside_maverick = f1_score_weighted_difference_computed_without_mav_label.max()
    maverick_gain_label = f1_score_weighted_difference_computed.iloc[maverick_label]

    value_distance_dataframe["MaverickWeightedOnlyLabelGain"] = [maverick_gain_label]
    value_distance_dataframe["MaverickWeightedMaxLoss"] = [max_loss]
    value_distance_dataframe["MaverickWeightedAvgLoss"] = [avg_loss]
    value_distance_dataframe["MaverickWeightedMaxGainBesidesOnlyLabel"] = [max_gain_beside_maverick]


    return value_distance_dataframe

In [78]:
f1_score_maverick_comparisons = pd.DataFrame()

training_configurations_mavericks = [scenario for scenario in training_configurations_for_distances
                                     if scenario["partition"] == "manual"
                                      and scenario.get("is_maverick_scenario", False) == True
                                      and scenario.get("exclude_maverick", False) == False
                                      and scenario.get("dataset") != "wids_dataset"]

for training_configuration in training_configurations_mavericks:
    aggregation_process = training_configuration["strategy"]
    dataset_name = training_configuration["dataset"]
    maverick_number = training_configuration["maverick_client_number"]
    maverick_label = training_configuration["maverick_special_class_or_columns_for_subpopulation"]
    type_of_partition = training_configuration["partition"]
    additional_parameter = training_configuration["name"]
    # additional_parameter = configuration["Dirichlet/NamePartition"]
    random_seed = training_configuration["run_number"]

    f1_score_maverick_comparisons = pd.concat([f1_score_maverick_comparisons, f1_score_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick_number, maverick_label)], ignore_index=True)

In [79]:
def mse_subpopulation_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick_number, maverick_categorical_features, maverick_value):
    mse_evaluation_difference = pd.DataFrame()

    path_to_result_dataframes_with_maverick = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True, False)
    path_to_result_dataframes_without_maverick = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True, True)

    mse_with_maverick = pd.read_csv(path_to_result_dataframes_with_maverick + os.sep + "Evaluation" + os.sep + "Evaluation_Subpopulations.csv", index_col=[0, 1, 2, 3])
    mse_without_maverick = pd.read_csv(path_to_result_dataframes_without_maverick + os.sep + "Evaluation" + os.sep + "Evaluation_Subpopulations.csv", index_col=[0, 1, 2, 3])

    filtered_result_for_last_round_with_maverick = mse_with_maverick.loc[mse_with_maverick.index[-1][0], "Global", :, "MSE"].copy(deep=True)
    filtered_result_for_last_round_without_maverick = mse_without_maverick.loc[mse_without_maverick.index[-1][0], "Global", :, "MSE"].copy(deep=True)

    filtered_result_for_last_round_with_maverick.drop("Class", axis=1, inplace=True)
    filtered_result_for_last_round_without_maverick.drop("Class", axis=1, inplace=True)

    path_to_data = get_data_from_route(dataset_name, type_of_partition, additional_parameter, random_seed)

    categories_with_maverick = pd.DataFrame()
    categories_without_maverick = pd.DataFrame()

    number_of_clients = int(len(os.listdir(path_to_data)) / 4) - 1

    for client_number in range(number_of_clients):
        if client_number == maverick_number:
            categories_without_maverick = pd.concat([categories_without_maverick, pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_mav_data_excluded_X_test.csv", index_col=0).sum(axis=0)).T], ignore_index=True)
            categories_with_maverick = pd.concat([categories_with_maverick, pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_X_test.csv", index_col=0).sum(axis=0)).T], ignore_index=True)
        else:
            subpopulation_data = pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_X_test.csv", index_col=0).sum(axis=0)).T
            categories_with_maverick = pd.concat([categories_with_maverick, subpopulation_data], ignore_index=True)
            categories_without_maverick = pd.concat([categories_without_maverick, subpopulation_data], ignore_index=True)


    categories_with_maverick = categories_with_maverick.filter(regex="facility_type_.*", axis=1)
    categories_without_maverick = categories_without_maverick.filter(regex="facility_type_.*", axis=1)

    subpopulation_percentages_with_maverick = categories_with_maverick.sum(axis=0) / categories_with_maverick.to_numpy().sum()
    subpopulation_percentages_without_maverick = categories_without_maverick.sum(axis=0) / categories_without_maverick.to_numpy().sum()

    subpopulation_percentages_with_maverick.index = (
    subpopulation_percentages_with_maverick.index.str.replace('facility_type_', '', regex=False)
)
    subpopulation_percentages_without_maverick.index = (
    subpopulation_percentages_without_maverick.index.str.replace('facility_type_', '', regex=False)
)

    # Aggregated statistics with raw values
    mse_difference_computed = filtered_result_for_last_round_with_maverick - filtered_result_for_last_round_without_maverick
    mse_difference_computed_without_mav_label: pd.DataFrame = mse_difference_computed.copy()
    mse_difference_computed_without_mav_label.loc[maverick_value] = 0
    max_loss_gain = mse_difference_computed_without_mav_label.min().iat[0]
    avg_loss = mse_difference_computed_without_mav_label.mean().iat[0]
    max_loss_gain_beside_maverick = mse_difference_computed_without_mav_label.min().iat[0]
    max_impairment_by_maverick = mse_difference_computed_without_mav_label.max().iat[0]
    # Unsure why, but iat is needed to remove name
    maverick_gain_subpopulation = mse_difference_computed.loc[maverick_value].iat[0]

    mse_evaluation_difference["AggregationProcess"] = [aggregation_process]
    # mse_evaluation_difference["Dataset"] = [dataset_name]
    mse_evaluation_difference["Partition"] = [additional_parameter]
    # mse_evaluation_difference["Metric"] = ["MSE"]
    mse_evaluation_difference["RandomSeed"] = [random_seed]
    mse_evaluation_difference["MaverickOnlySubpopulationGain"] = [maverick_gain_subpopulation]
    mse_evaluation_difference["MaverickMaxLossGain"] = [max_loss_gain]
    mse_evaluation_difference["MaverickAvgLoss"] = [avg_loss]
    mse_evaluation_difference["MaverickMaxLossGainBesidesOnlyLabel"] = [max_loss_gain_beside_maverick]
    mse_evaluation_difference["MaverickMaxImpairment"] = [max_impairment_by_maverick]

    # display(filtered_result_for_last_round_with_maverick)
    # display(pd.Series(subpopulation_percentages_with_maverick, index=subpopulation_percentages_with_maverick.index))
    weighted_result_for_last_round_with_maverick = filtered_result_for_last_round_with_maverick.mul(subpopulation_percentages_with_maverick, axis=0)
    weighted_result_for_last_round_without_maverick = filtered_result_for_last_round_without_maverick.mul( subpopulation_percentages_without_maverick, axis=0)
    mse_weighted_difference_computed = weighted_result_for_last_round_with_maverick - weighted_result_for_last_round_without_maverick
    mse_weighted_difference_computed_without_mav_label: pd.DataFrame = mse_weighted_difference_computed.copy()
    mse_weighted_difference_computed_without_mav_label.loc[maverick_value] = 0
    max_loss_gain = mse_weighted_difference_computed_without_mav_label.min().iat[0]
    avg_loss = mse_weighted_difference_computed_without_mav_label.mean().iat[0]
    max_loss_gain_beside_maverick = mse_weighted_difference_computed_without_mav_label.min().iat[0]
    max_impairment_by_maverick = mse_weighted_difference_computed_without_mav_label.max().iat[0]
    # Unsure why, but iat is needed to remove name
    maverick_gain_subpopulation = mse_weighted_difference_computed.loc[maverick_value].iat[0]

    mse_evaluation_difference["MaverickWeightedOnlySubpopulationGain"] = [maverick_gain_subpopulation]
    mse_evaluation_difference["MaverickWeightedMaxLossGain"] = [max_loss_gain]
    mse_evaluation_difference["MaverickWeightedAvgLoss"] = [avg_loss]
    mse_evaluation_difference["MaverickWeightedMaxLossGainBesidesOnlyLabel"] = [max_loss_gain_beside_maverick]
    mse_evaluation_difference["MaverickWeightedMaxImpairment"] = [max_impairment_by_maverick]

    return mse_evaluation_difference

In [80]:
mse_subpopulation_maverick_comparisons= pd.DataFrame()

training_configurations_mavericks = [scenario for scenario in training_configurations_for_distances
                                     if scenario["partition"] == "manual"
                                      and scenario.get("is_maverick_scenario", False) == True
                                      and scenario.get("exclude_maverick", False) == False
                                      and scenario.get("dataset") == "wids_dataset"]

for training_configuration in training_configurations_mavericks:
    aggregation_process = training_configuration["strategy"]
    dataset_name = training_configuration["dataset"]
    type_of_partition = training_configuration["partition"]
    additional_parameter = training_configuration["name"]
    random_seed = training_configuration["run_number"]

    maverick_number = training_configuration["maverick_client_number"]
    maverick_columns_for_subpopulation = training_configuration["maverick_special_class_or_columns_for_subpopulation"]
    maverick_category_value = training_configuration["maverick_category_value"]

    mse_subpopulation_maverick_comparisons = pd.concat([mse_subpopulation_maverick_comparisons, mse_subpopulation_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick_number, maverick_columns_for_subpopulation, maverick_category_value)], ignore_index=True)

## Computing distances and value for the different experiments

In [81]:
def centralized_decentralized_evaluation_difference(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, exclude_maverick):
    path_to_result_dataframes = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, exclude_maverick)

    centralized_decentralized_evaluation_difference = pd.DataFrame(columns=["AggregationProcess", "Dataset", "AdditionalParameter", "RandomSeed", "Metric", "Difference_in_value"])
    metrics = []
    difference_in_value = []

    for file in os.listdir(path_to_result_dataframes + os.sep + "Evaluation"):
        results_dataframe = pd.read_csv(path_to_result_dataframes + os.sep + "Evaluation" + os.sep + file, index_col=0)
        if "F1Score" == file or "Subpopulation" in file:
            pass
        else:
            metrics.append(file.split('_')[1])
            difference_in_value.append(abs(float(results_dataframe.iloc[-1, results_dataframe.columns.get_loc('Global')]) - float(results_dataframe.iloc[-1, results_dataframe.columns.get_loc("Aggregated")])))

    centralized_decentralized_evaluation_difference["AggregationProcess"] = [aggregation_process for _ in range(len(metrics))]
    centralized_decentralized_evaluation_difference["Dataset"] = [dataset_name for _ in range(len(metrics))]
    centralized_decentralized_evaluation_difference["TypePartition"] = [type_of_partition for _ in range(len(metrics))]
    centralized_decentralized_evaluation_difference["AdditionalParameter"] = [additional_parameter for _ in range(len(metrics))]
    centralized_decentralized_evaluation_difference["RandomSeed"] = [random_seed for _ in range(len(metrics))]
    centralized_decentralized_evaluation_difference["Metric"] = metrics
    centralized_decentralized_evaluation_difference["Difference_in_value"] = difference_in_value

    return centralized_decentralized_evaluation_difference

In [82]:
differences_in_evaluation = pd.DataFrame()

for configuration in training_configurations_for_distances:
    aggregation_process = configuration["strategy"]
    dataset_name = configuration["dataset"]
    type_of_partition = configuration["partition"]
    if type_of_partition == "dirichlet":
        additional_parameter = configuration["alpha"]
        is_maverick_scenario = False
        exclude_maverick = False
    elif type_of_partition == "manual":
        additional_parameter = configuration["name"]
        is_maverick_scenario = configuration.get("is_maverick_scenario", False)
        exclude_maverick = configuration.get("exclude_maverick", False)
    else:
        raise NotImplementedError
    random_seed = configuration["run_number"]
    # additional_parameter = configuration["Dirichlet/NamePartition"]
    differences_in_evaluation = pd.concat([differences_in_evaluation, centralized_decentralized_evaluation_difference(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, exclude_maverick)], ignore_index=True)

In [83]:
display(differences_in_evaluation.groupby(["Metric"]).mean(numeric_only=True).T)

Metric,Accuracy,CrossEntropyLoss,F1Score,F1ScoreMacro,F1ScoreMicro,MAE,MCC,MSE,RMSE
RandomSeed,3.0,3.000000e+00,3.0,3.0,3.0,3.000000,3.0,3.000000e+00,3.000000e+00
Difference_in_value,0.0,4.945815e-10,0.0,0.0,0.0,0.159482,0.0,5.443223e-07,6.526860e-09


In [84]:
def compute_spearman_correlation_distance(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, type_of_loss):
    path_to_distance_dataframes = get_distances_from_route(dataset_name, type_of_partition, additional_parameter, random_seed)
    path_to_results_dataframes = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario)

    spearman_rank_sv_metric_centralized_decentralized = pd.DataFrame()
    distances = []
    spearman_values = []
    spearman_p_values = []

    # print(f"Dataset name: {dataset_name}")
    # print(f"Type_of_partition name: {type_of_partition}")
    # print(f"Additional_parameter name: {additional_parameter}")

    if type_of_loss == "MSE":
        results_dataframe = pd.read_csv(path_to_results_dataframes + os.sep + "Shapley_Value" + os.sep + "SV_MSE")
    else:
        results_dataframe = pd.read_csv(path_to_results_dataframes + os.sep + "Shapley_Value" + os.sep + "SV_CrossEntropyLoss")
    
    grouped_sv = results_dataframe.groupby(["Evaluator"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Round"], axis=1)
    grouped_sv.drop("Aggregated", inplace=True)

    # display(grouped_sv)

    for file in os.listdir(path_to_distance_dataframes):
        distances.append(file.split('.')[0])
        if file == "volume.csv":
            volume_series = pd.read_csv(path_to_distance_dataframes + os.sep + file, index_col=0)
            # volume_series.drop("Global", inplace=True)
            def statistic(x): # permute only `x`
                return scipy.stats.spearmanr(x, grouped_sv.loc["Centralized"]).statistic
            res_exact = scipy.stats.permutation_test((volume_series.loc[:],), statistic,
                permutation_type='pairings')
            spearman_values.append(res_exact.statistic[0])
            spearman_p_values.append(res_exact.pvalue[0])

        else:
            distance_dataframe = pd.read_csv(path_to_distance_dataframes + os.sep + file, index_col=0)
            spearman_list = []
            pvalue_list = []
            for index, row in grouped_sv.iterrows():
                if index == "Centralized":
                    # def statistic(x): # permute only `x`
                    #     return scipy.stats.spearmanr(x, grouped_sv.loc["Centralized"]).statistic
                    # res_exact = scipy.stats.permutation_test((distance_dataframe.loc[:, "Global"],), statistic,
                    #     permutation_type='pairings')
                    try:
                        res_exact = scipy.stats.spearmanr(grouped_sv.loc["Centralized"], distance_dataframe.loc[:, "Global"])
                    except:
                        print(f"Dataset name: {dataset_name}")
                        print(f"Type_of_partition name: {type_of_partition}")
                        print(f"Additional_parameter name: {additional_parameter}")
                        print(f"Random seed: {random_seed}")
                        print(f"Grouped: {grouped_sv.loc[str(index)]}")
                        print(f"Distance dataframe: {distance_dataframe.loc[:, str(index)]}")
                    spearman_list.append(res_exact.statistic)
                    pvalue_list.append(res_exact.pvalue)
                else:
                    # def statistic(x): # permute only `x`
                    #     return scipy.stats.spearmanr(x, grouped_sv.loc[str(index)]).statistic
                    # res_exact = scipy.stats.permutation_test((distance_dataframe.loc[:, str(index)],), statistic,
                    #     permutation_type='pairings')
                    try:
                        res_exact = scipy.stats.spearmanr(grouped_sv.loc[str(index)], distance_dataframe.loc[:, str(index)])
                    except:
                        print(f"Dataset name: {dataset_name}")
                        print(f"Type_of_partition name: {type_of_partition}")
                        print(f"Additional_parameter name: {additional_parameter}")
                        print(f"Random seed: {random_seed}")
                        print(f"Grouped: {grouped_sv.loc[str(index)]}")
                        print(f"Distance dataframe: {distance_dataframe.loc[:, str(index)]}")
                    spearman_list.append(res_exact.statistic)
                    pvalue_list.append(res_exact.pvalue)

            # Fisher transformation for averaging correlations.
            spearman_values.append(np.tanh(np.mean(np.arctanh(spearman_list))))
            spearman_p_values.append(np.mean(pvalue_list))

    spearman_rank_sv_metric_centralized_decentralized["Dataset"] = [dataset_name for _ in range(len(distances))]
    spearman_rank_sv_metric_centralized_decentralized["TypePartition"] = [type_of_partition for _ in range(len(distances))]
    spearman_rank_sv_metric_centralized_decentralized["AdditionalParameter"] = [additional_parameter for _ in range(len(distances))]
    spearman_rank_sv_metric_centralized_decentralized["RandomSeed"] = [random_seed for _ in range(len(distances))]
    spearman_rank_sv_metric_centralized_decentralized["Distance"] = distances
    spearman_rank_sv_metric_centralized_decentralized["SpearmanCorrelation"] = spearman_values
    spearman_rank_sv_metric_centralized_decentralized["p-value-spearman"] = spearman_p_values

    return spearman_rank_sv_metric_centralized_decentralized

In [85]:
def compute_pearson_correlation_distance(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, type_of_loss="CrossEntropyLoss"):
    path_to_distance_dataframes = get_distances_from_route(dataset_name, type_of_partition, additional_parameter, random_seed)
    path_to_results_dataframes = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario)

    pearson_sv_metric_centralized_decentralized = pd.DataFrame()
    distances = []
    pearson_values = []
    pearson_p_values = []
    variance_scenario = []

    # print(f"Dataset name: {dataset_name}")
    # print(f"Type_of_partition name: {type_of_partition}")
    # print(f"Additional_parameter name: {additional_parameter}")

    if type_of_loss == "MSE":
        results_dataframe = pd.read_csv(path_to_results_dataframes + os.sep + "Shapley_Value" + os.sep + "SV_MSE")
    else:
        results_dataframe = pd.read_csv(path_to_results_dataframes + os.sep + "Shapley_Value" + os.sep + "SV_CrossEntropyLoss")
        
    grouped_sv = results_dataframe.groupby(["Evaluator"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Round"], axis=1)
    grouped_sv.drop("Aggregated", inplace=True)

    # display(grouped_sv)

    for file in os.listdir(path_to_distance_dataframes):
        distances.append(file.split('.')[0])
        if file == "volume.csv":
            volume_series = pd.read_csv(path_to_distance_dataframes + os.sep + file, index_col=0)
            # volume_series.drop("Global", inplace=True)
            res_exact = scipy.stats.pearsonr(grouped_sv.loc["Centralized"], volume_series.to_numpy().reshape(-1))
            pearson_values.append(res_exact.statistic)
            pearson_p_values.append(res_exact.pvalue)
            # variance_scenario.append(np.var(grouped_sv.loc["Centralized"]))
        else:
            distance_dataframe = pd.read_csv(path_to_distance_dataframes + os.sep + file, index_col=0)
            if file == "yShiftDataframe.csv":
                distance_dataframe.replace([np.inf, -np.inf], 0, inplace=True)
                distance_dataframe.replace([np.NAN], 0, inplace=True)
            pearsonr_list = []
            pvalue_list = []
            for index, row in grouped_sv.iterrows():
                if index == "Centralized":
                    res_exact = scipy.stats.pearsonr(grouped_sv.loc["Centralized"], distance_dataframe.loc[:, "Global"])
                    pearsonr_list.append(res_exact.statistic)
                    pvalue_list.append(res_exact.pvalue)
                else:            
                    res_exact = scipy.stats.pearsonr(grouped_sv.loc[str(index)], distance_dataframe.loc[:, str(index)])
                    pearsonr_list.append(res_exact.statistic)
                    pvalue_list.append(res_exact.pvalue)

            # variance_scenario.append(np.var(grouped_sv.loc["Centralized"]))
            # Fisher transformation for averaging correlations.
            pearson_values.append(np.tanh(np.mean(np.arctanh(pearsonr_list))))
            pearson_p_values.append(np.mean(pvalue_list))


    pearson_sv_metric_centralized_decentralized["Dataset"] = [dataset_name for _ in range(len(distances))]
    pearson_sv_metric_centralized_decentralized["TypePartition"] = [type_of_partition for _ in range(len(distances))]
    pearson_sv_metric_centralized_decentralized["AdditionalParameter"] = [additional_parameter for _ in range(len(distances))]
    pearson_sv_metric_centralized_decentralized["RandomSeed"] = [random_seed for _ in range(len(distances))]
    pearson_sv_metric_centralized_decentralized["Distance"] = distances
    pearson_sv_metric_centralized_decentralized["PearsonCorrelation"] = pearson_values
    pearson_sv_metric_centralized_decentralized["p-value-pearson"] = pearson_p_values
    # pearson_sv_metric_centralized_decentralized["VarianceSV"] = variance_scenario

    return pearson_sv_metric_centralized_decentralized

In [86]:
spearman_rank_sv_distance = pd.DataFrame()
pearson_rank_sv_distance = pd.DataFrame()


for configuration in training_configurations_for_distances:
    aggregation_process = configuration["strategy"]
    dataset_name = configuration["dataset"]
    type_of_partition = configuration["partition"]
    if type_of_partition == "dirichlet":
        additional_parameter = configuration["alpha"]
        is_maverick_scenario = False
        exclude_maverick = False
    elif type_of_partition == "manual":
        additional_parameter = configuration["name"]
        is_maverick_scenario = configuration.get("is_maverick_scenario", False)
        exclude_maverick = configuration.get("exclude_maverick", False)
    else:
        raise NotImplementedError
    random_seed = configuration["run_number"]

    if not exclude_maverick:
        if dataset_name == "wids_dataset":
            type_of_loss = "MSE"
        else:
            type_of_loss = "CrossEntropyLoss"
        # additional_parameter = configuration["Dirichlet/NamePartition"]
        spearman_rank_sv_distance = pd.concat([spearman_rank_sv_distance, compute_spearman_correlation_distance(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed,is_maverick_scenario, type_of_loss)], ignore_index=True)
        pearson_rank_sv_distance = pd.concat([pearson_rank_sv_distance, compute_pearson_correlation_distance(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, type_of_loss)], ignore_index=True)

correlation_sv_distance = spearman_rank_sv_distance.copy()
correlation_sv_distance["PearsonCorrelation"] = pearson_rank_sv_distance["PearsonCorrelation"]
correlation_sv_distance["p-value-pearson"] = pearson_rank_sv_distance["p-value-pearson"]
# correlation_sv_distance["VarianceSV"] = pearson_rank_sv_distance["VarianceSV"]

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divid

In [87]:
def flattened_spearman_correlation_between_value_and_distance(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, type_of_loss="CrossEntropyLoss"):
    path_to_distance_dataframes = get_distances_from_route(dataset_name, type_of_partition, additional_parameter, random_seed)
    path_to_results_dataframes = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario)

    spearman_rank_sv_metric_centralized_decentralized = pd.DataFrame()
    distances = []
    spearman_values = []
    spearman_p_values = []

    # print(f"Dataset name: {dataset_name}")
    # print(f"Type_of_partition name: {type_of_partition}")
    # print(f"Additional_parameter name: {additional_parameter}")

    if type_of_loss == "MSE":
        results_dataframe = pd.read_csv(path_to_results_dataframes + os.sep + "Shapley_Value" + os.sep + "SV_MSE")
    else:
        results_dataframe = pd.read_csv(path_to_results_dataframes + os.sep + "Shapley_Value" + os.sep + "SV_CrossEntropyLoss")

    grouped_sv = results_dataframe.groupby(["Evaluator"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Round"], axis=1)
    grouped_sv.drop("Aggregated", inplace=True)

    # display(grouped_sv)

    for file in os.listdir(path_to_distance_dataframes):
        distances.append(file.split('.')[0])
        if file == "volume.csv":
            volume_series = pd.read_csv(path_to_distance_dataframes + os.sep + file, index_col=0)
            # volume_series.drop("Global", inplace=True)
            def statistic(x): # permute only `x`
                return scipy.stats.spearmanr(x, grouped_sv.loc["Centralized"]).statistic
            res_exact = scipy.stats.permutation_test((volume_series.loc[:],), statistic,
                permutation_type='pairings')
            spearman_values.append(res_exact.statistic[0])
            spearman_p_values.append(res_exact.pvalue[0])

        else:
            distance_dataframe = pd.read_csv(path_to_distance_dataframes + os.sep + file, index_col=0)
            spearman_list = []
            pvalue_list = []
            for index, row in grouped_sv.iterrows():
                if index == "Centralized":
                    # def statistic(x): # permute only `x`
                    #     return scipy.stats.spearmanr(x, grouped_sv.loc["Centralized"]).statistic
                    # res_exact = scipy.stats.permutation_test((distance_dataframe.loc[:, "Global"],), statistic,
                    #     permutation_type='pairings')
                    res_exact = scipy.stats.spearmanr(grouped_sv.loc["Centralized"], distance_dataframe.loc[:, "Global"])
                    spearman_list.append(res_exact.statistic)
                    pvalue_list.append(res_exact.pvalue)
                else:
                    # def statistic(x): # permute only `x`
                    #     return scipy.stats.spearmanr(x, grouped_sv.loc[str(index)]).statistic
                    # res_exact = scipy.stats.permutation_test((distance_dataframe.loc[:, str(index)],), statistic,
                    #     permutation_type='pairings')
                    res_exact = scipy.stats.spearmanr(grouped_sv.loc[str(index)], distance_dataframe.loc[:, str(index)])
                    spearman_list.append(res_exact.statistic)
                    pvalue_list.append(res_exact.pvalue)

            # Fisher transformation for averaging correlations.
            spearman_values.append(np.tanh(np.mean(np.arctanh(spearman_list))))
            spearman_p_values.append(np.mean(pvalue_list))

    spearman_rank_sv_metric_centralized_decentralized["Dataset"] = [dataset_name for _ in range(len(distances))]
    spearman_rank_sv_metric_centralized_decentralized["TypePartition"] = [type_of_partition for _ in range(len(distances))]
    spearman_rank_sv_metric_centralized_decentralized["AdditionalParameter"] = [additional_parameter for _ in range(len(distances))]
    spearman_rank_sv_metric_centralized_decentralized["RandomSeed"] = [random_seed for _ in range(len(distances))]
    spearman_rank_sv_metric_centralized_decentralized["Distance"] = distances
    spearman_rank_sv_metric_centralized_decentralized["SpearmanCorrelation"] = spearman_values
    spearman_rank_sv_metric_centralized_decentralized["p-value-spearman"] = spearman_p_values

    return spearman_rank_sv_metric_centralized_decentralized

In [88]:
display(spearman_rank_sv_distance[spearman_rank_sv_distance["Distance"] == "hellinger"])

,Dataset,TypePartition,AdditionalParameter,RandomSeed,Distance,SpearmanCorrelation,p-value-spearman
2,wids_dataset,manual,wids_energy_non_iid_by_label,1,hellinger,-0.171248,0.496633
9,wids_dataset,manual,wids_energy_iid_sampling,1,hellinger,0.344856,0.351314
16,wids_dataset,manual,wids_energy_feature_skew_building_type,1,hellinger,-0.462907,0.191394
23,wids_dataset,manual,wids_energy_feature_skew_state_factor,1,hellinger,-0.372868,0.511034
30,wids_dataset,manual,wids_energy_maverick_facility_type_grocery_store,1,hellinger,-0.308397,0.624886
...,...,...,...,...,...,...,...
2272,edge-iot-coreset,dirichlet,1000,5,hellinger,-1.000000,0.037584
2280,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,5,hellinger,-0.463900,0.244642
2288,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Only_Normal,5,hellinger,-0.707107,0.181690
2296,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_sql_injection,5,hellinger,-0.528155,0.181690


In [89]:
display(correlation_sv_distance)

,Dataset,TypePartition,AdditionalParameter,RandomSeed,Distance,SpearmanCorrelation,p-value-spearman,PearsonCorrelation,p-value-pearson
0,wids_dataset,manual,wids_energy_non_iid_by_label,1,yShiftDataframe,0.102644,0.379622,-0.022818,0.476289
1,wids_dataset,manual,wids_energy_non_iid_by_label,1,relevance,0.323940,0.367727,0.177042,0.642622
2,wids_dataset,manual,wids_energy_non_iid_by_label,1,hellinger,-0.171248,0.496633,0.090358,0.731701
3,wids_dataset,manual,wids_energy_non_iid_by_label,1,diversity,-0.539397,0.202182,-0.440907,0.486689
4,wids_dataset,manual,wids_energy_non_iid_by_label,1,wasserstein,-0.770930,0.146852,-0.788348,0.211226
...,...,...,...,...,...,...,...,...,...
2305,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_ddos_udp,5,diversity,-0.481862,0.436031,-0.658945,0.298097
2306,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_ddos_udp,5,negativeConditionalEntropy,0.223723,0.466188,0.579582,0.346254
2307,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_ddos_udp,5,wasserstein,-0.998360,0.224394,-0.614555,0.115512
2308,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_ddos_udp,5,gaussian_mmd,-0.072832,0.496302,-0.531308,0.131217


In [90]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
np.random.seed(1)

#garbage graph
fig = px.scatter(x=[0, 1, 2, 3, 4], y=[0, 1, 4, 9, 16])
fig.show()
fig.write_image("random.pdf")

/home/jose/miniconda3/envs/flower_py3-10/lib/python3.10/site-packages/plotly/io/_renderers.py:395: DeprecationWarning:

distutils Version classes are deprecated. Use packaging.version instead.



In [91]:
distance_dictionary = {
    "performanceDegradation": "Performance Degradation (I)",
    "yShiftDataframe": "X->Y Shift (I)",
    "relevance": "Relevance (D)",
    "diversity": "Diversity (D)",
    "hellinger": "Hellinger Distance (I)",
    "wasserstein": "FedBary Distance (I)",
    "gaussian_mmd": "MMD - Gaussian Kernel (I)",
    "volume": "Volume of data (D)"
}

fig = go.Figure()
for metric, name_of_metric in distance_dictionary.items():
    sliced_df = correlation_sv_distance[correlation_sv_distance["Distance"] == metric]
    fig.add_trace(go.Box(y=sliced_df["PearsonCorrelation"], name=name_of_metric))


fig.update_layout(title_text="Pearson Correlation between Distance and Value", showlegend=False, template="plotly_white")

fig.show()

fig.write_image("images/distance_value_pearson_correlation.pdf")

In [92]:
spearman_rank_sv_distance = pd.DataFrame()
pearson_rank_sv_distance = pd.DataFrame()

training_configurations_mavericks = [scenario for scenario in training_configurations_for_distances
                                    if scenario["partition"] == "manual" and scenario.get("is_maverick_scenario", False) == True and scenario.get("exclude_maverick", False) == False]

for configuration in training_configurations_mavericks:
    aggregation_process = configuration["strategy"]
    dataset_name = configuration["dataset"]
    type_of_partition = configuration["partition"]
    if type_of_partition == "dirichlet":
        additional_parameter = configuration["alpha"]
        is_maverick_scenario = False
        exclude_maverick = False
    elif type_of_partition == "manual":
        additional_parameter = configuration["name"]
        is_maverick_scenario = configuration.get("is_maverick_scenario", False)
        exclude_maverick = configuration.get("exclude_maverick", False)
    else:
        raise NotImplementedError
    random_seed = configuration["run_number"]

    if is_maverick_scenario and not exclude_maverick:
        if dataset_name == "wids_dataset":
            type_of_loss = "MSE"
        else:
            type_of_loss = "CrossEntropyLoss"
        # additional_parameter = configuration["Dirichlet/NamePartition"]
        spearman_rank_sv_distance = pd.concat([spearman_rank_sv_distance, compute_spearman_correlation_distance(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed,is_maverick_scenario, type_of_loss)], ignore_index=True)
        pearson_rank_sv_distance = pd.concat([pearson_rank_sv_distance, compute_pearson_correlation_distance(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, is_maverick_scenario, type_of_loss)], ignore_index=True)

correlation_sv_distance_only_mavericks = spearman_rank_sv_distance.copy()
correlation_sv_distance_only_mavericks["PearsonCorrelation"] = pearson_rank_sv_distance["PearsonCorrelation"]
correlation_sv_distance_only_mavericks["p-value-pearson"] = pearson_rank_sv_distance["p-value-pearson"]

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divide by zero encountered in arctanh

/tmp/ipykernel_6244/3397378757.py:75: RuntimeWarning:

divid

In [93]:
correlation_sv_distance_only_mavericks[correlation_sv_distance_only_mavericks["Distance"] == "wasserstein"]

,Dataset,TypePartition,AdditionalParameter,RandomSeed,Distance,SpearmanCorrelation,p-value-spearman,PearsonCorrelation,p-value-pearson
4,wids_dataset,manual,wids_energy_maverick_facility_type_grocery_store,1,wasserstein,-0.344127,0.524348,-0.275969,0.596274
11,wids_dataset,manual,wids_energy_maverick_facility_type_uncategoriz...,1,wasserstein,-0.522379,0.399494,-0.505535,0.462156
18,wids_dataset,manual,wids_energy_maverick_facility_type_grocery_store,2,wasserstein,0.204595,0.606494,0.173088,0.636436
25,wids_dataset,manual,wids_energy_maverick_facility_type_uncategoriz...,2,wasserstein,0.028814,0.425456,-0.141276,0.341000
32,wids_dataset,manual,wids_energy_maverick_facility_type_grocery_store,3,wasserstein,-0.056467,0.336080,-0.077052,0.346770
...,...,...,...,...,...,...,...,...,...
1135,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_ddos_udp,4,wasserstein,-0.472465,0.476432,-0.542452,0.180721
1143,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Least_Class,5,wasserstein,-0.035791,0.521702,0.224093,0.469280
1151,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_Only_Normal,5,wasserstein,-0.998218,0.409109,-0.999700,0.000011
1159,edge-iot-coreset,manual,edgeiot_coreset_1_Maverick_sql_injection,5,wasserstein,-0.505771,0.283936,-0.645939,0.217533


In [94]:
import plotly.graph_objects as go
import numpy as np
np.random.seed(1)

distance_dictionary = {
    "performanceDegradation": "Performance Degradation (I)",
    "yShiftDataframe": "X->Y Shift (I)",
    "relevance": "Relevance (D)",
    "diversity": "Diversity (D)",
    "hellinger": "Hellinger Distance (I)",
    "wasserstein": "FedBary Distance (I)",
    "gaussian_mmd": "MMD - Gaussian Kernel (I)",
    "volume": "Volume of data (D)"
}

fig1 = go.Figure()
for metric, name_of_metric in distance_dictionary.items():
    sliced_df = correlation_sv_distance_only_mavericks[correlation_sv_distance_only_mavericks["Distance"] == metric]
    fig1.add_trace(go.Box(y=sliced_df["PearsonCorrelation"], name=name_of_metric))


fig1.update_layout(title_text="Pearson Correlation between Distance and Value only Mavericks", showlegend=False, template="plotly_white")

fig1.show()

fig1.write_image("images/distance_value_only_mavericks_pearson_correlation.pdf")

/home/jose/miniconda3/envs/flower_py3-10/lib/python3.10/site-packages/plotly/io/_renderers.py:395: DeprecationWarning:

distutils Version classes are deprecated. Use packaging.version instead.



In [95]:
display(correlation_sv_distance[correlation_sv_distance["Dataset"] == "electric-consumption"])

,Dataset,TypePartition,AdditionalParameter,RandomSeed,Distance,SpearmanCorrelation,p-value-spearman,PearsonCorrelation,p-value-pearson


In [96]:
display(correlation_sv_distance.groupby(["Dataset", "Distance"]).mean(numeric_only=True))

RandomSeed  SpearmanCorrelation  \
Dataset          Distance                                                      
edge-iot-coreset diversity                          3.0            -0.448950   
                 gaussian_mmd                       3.0            -0.524909   
                 hellinger                          3.0            -0.570688   
                 negativeConditionalEntropy         3.0             0.225318   
                 relevance                          3.0             0.348085   
                 volume                             3.0             0.359048   
                 wasserstein                        3.0            -0.442070   
                 yShiftDataframe                    3.0             0.180087   
har              diversity                          3.0            -0.394895   
                 gaussian_mmd                       3.0            -0.164290   
                 hellinger                          3.0            -0.616158   
                 negativeConditionalEntropy         3.0             0.112167   
                 relevance                          3.0             0.281336   
                 volume                             3.0             0.228844   
                 wasserstein                        3.0            -0.260262   
                 yShiftDataframe                    3.0             0.104570   
wids_dataset     diversity                          3.0            -0.377368   
                 gaussian_mmd                       3.0            -0.346102   
                 hellinger                          3.0            -0.096285   
                 relevance                          3.0             0.252783   
                 volume                             3.0             0.228571   
                 wasserstein                        3.0            -0.406663   
                 yShiftDataframe                    3.0            -0.007085   

                                             p-value-spearman  \
Dataset          Distance                                       
edge-iot-coreset diversity                           0.423364   
                 gaussian_mmd                        0.401767   
                 hellinger                           0.221225   
                 negativeConditionalEntropy          0.479065   
                 relevance                           0.441644   
                 volume                              0.502063   
                 wasserstein                         0.411769   
                 yShiftDataframe                     0.470453   
har              diversity                           0.442419   
                 gaussian_mmd                        0.467562   
                 hellinger                           0.206317   
                 negativeConditionalEntropy          0.482943   
                 relevance                           0.469407   
                 volume                              0.491852   
                 wasserstein                         0.455064   
                 yShiftDataframe                     0.484873   
wids_dataset     diversity                           0.359044   
                 gaussian_mmd                        0.333671   
                 hellinger                           0.499967   
                 relevance                           0.419329   
                 volume                              0.387963   
                 wasserstein                         0.336402   
                 yShiftDataframe                     0.488324   

                                             PearsonCorrelation  \
Dataset          Distance                                         
edge-iot-coreset diversity                            -0.594943   
                 gaussian_mmd                         -0.611647   
                 hellinger                            -0.711239   
                 negativeConditionalEntropy            0.518704   
                

In [97]:
display(correlation_sv_distance.groupby(["TypePartition", "Distance"]).mean(numeric_only=True))

RandomSeed  SpearmanCorrelation  \
TypePartition Distance                                                      
dirichlet     diversity                          3.0            -0.517783   
              gaussian_mmd                       3.0            -0.590836   
              hellinger                          3.0            -0.752782   
              negativeConditionalEntropy         3.0             0.518435   
              relevance                          3.0             0.379983   
              volume                             3.0             0.249683   
              wasserstein                        3.0            -0.425344   
              yShiftDataframe                    3.0             0.195416   
manual        diversity                          3.0            -0.361744   
              gaussian_mmd                       3.0            -0.239713   
              hellinger                          3.0            -0.312067   
              negativeConditionalEntropy         3.0            -0.093527   
              relevance                          3.0             0.260196   
              volume                             3.0             0.284898   
              wasserstein                        3.0            -0.343160   
              yShiftDataframe                    3.0             0.055761   

                                          p-value-spearman  \
TypePartition Distance                                       
dirichlet     diversity                           0.395630   
              gaussian_mmd                        0.357884   
              hellinger                           0.236564   
              negativeConditionalEntropy          0.423970   
              relevance                           0.424835   
              volume                              0.490679   
              wasserstein                         0.420722   
              yShiftDataframe                     0.468353   
manual        diversity                           0.417212   
              gaussian_mmd                        0.424288   
              hellinger                           0.326658   
              negativeConditionalEntropy          0.523780   
              relevance                           0.453166   
              volume                              0.452937   
              wasserstein                         0.397279   
              yShiftDataframe                     0.486192   

                                          PearsonCorrelation  p-value-pearson  
TypePartition Distance                                                         
dirichlet     diversity                            -0.499288         0.330994  
              gaussian_mmd                         -0.534691         0.312807  
              hellinger                            -0.748852         0.153975  
              negativeConditionalEntropy            0.546582         0.326225  
              relevance                             0.301729         0.372126  
              volume                                0.275129         0.437509  
              wasserstein                          -0.427908         0.377694  
              yShiftDataframe                       0.138498         0.428748  
manual        diversity                            -0.512978         0.293323  
              gaussian_mmd                         -0.533781         0.366232  
              hellinger                            -0.495196         0.249288  
              negativeConditionalEntropy            0.412611         0.456017  
              relevance                             0.362052         0.357337  
              volume                                0.437960         0.286632  
              wasserstein                          -0.527444         0.239767  
              yShiftDataframe                       0.101383         0.429763

In [98]:
display(correlation_sv_distance.groupby(["Dataset", "TypePartition", "Distance"]).mean(numeric_only=True))

RandomSeed  \
Dataset          TypePartition Distance                                 
edge-iot-coreset dirichlet     diversity                          3.0   
                               gaussian_mmd                       3.0   
                               hellinger                          3.0   
                               negativeConditionalEntropy         3.0   
                               relevance                          3.0   
                               volume                             3.0   
                               wasserstein                        3.0   
                               yShiftDataframe                    3.0   
                 manual        diversity                          3.0   
                               gaussian_mmd                       3.0   
                               hellinger                          3.0   
                               negativeConditionalEntropy         3.0   
                               relevance                          3.0   
                               volume                             3.0   
                               wasserstein                        3.0   
                               yShiftDataframe                    3.0   
har              dirichlet     diversity                          3.0   
                               gaussian_mmd                       3.0   
                               hellinger                          3.0   
                               negativeConditionalEntropy         3.0   
                               relevance                          3.0   
                               volume                             3.0   
                               wasserstein                        3.0   
                               yShiftDataframe                    3.0   
                 manual        diversity                          3.0   
                               gaussian_mmd                       3.0   
                               hellinger                          3.0   
                               negativeConditionalEntropy         3.0   
                               relevance                          3.0   
                               volume                             3.0   
                               wasserstein                        3.0   
                               yShiftDataframe                    3.0   
wids_dataset     manual        diversity                          3.0   
                               gaussian_mmd                       3.0   
                               hellinger                          3.0   
                               relevance                          3.0   
                               volume                             3.0   
                               wasserstein                        3.0   
                               yShiftDataframe                    3.0   

                                                           SpearmanCorrelation  \
Dataset          TypePartition Distance                                          
edge-iot-coreset dirichlet     diversity                             -0.628108   
                               gaussian_mmd                          -0.738978   
                               hellinger                             -0.878339   
                               negativeConditionalEntropy             0.619477   
                               relevance                              0.495778   
                               volume                                 0.482222   
                               wasserstein                           -0.643291   
                               yShiftDataframe                        0.270274   
                 manual        diversity                             -0.314581   
                               gaussian_mmd                          -0.364357   
                               hellinger                             -0.339950   

In [99]:
def ranking_Mavericks(aggregation_method, dataset_name, type_of_partition, additional_parameter):
    # distances = ["wasserstein.csv", "gaussian_mmd.csv", "negativeConditionalEntropy.csv", "hellinger.csv", "performanceDegradation.csv", "yShiftDataframe.csv", "relevance.csv", "diversity.csv"]
    # singleValue = ["volume.csv"]
    maverick_ranked = pd.DataFrame()

    path_to_dataframes = get_data_from_route(dataset_name, type_of_partition, additional_parameter, random_seed)
    num_clients = int(len(os.listdir(path_to_dataframes)) / 4) - 1
    # print(num_clients)
    path_to_result_dataframes = get_results_from_route(aggregation_method, dataset_name, type_of_partition, additional_parameter, random_seed, True, False)
    # path_to_result_dataframes = path_to_result_dataframes + os.sep + "Shapley_Values"
    ranked_sv_maverick = []
    clients_sv = {k: [] for k in range(num_clients)}
    sv_metric = []
    sv_metric_c_a = []
    # spearman_sv_distance_spearman_values = []
    # spearman_sv_distance_p_values = []

    for sv_file in os.listdir(path_to_result_dataframes + os.sep + "Shapley_Value"):
        results_dataframe = pd.read_csv(path_to_result_dataframes + os.sep + "Shapley_Value" + os.sep + sv_file)
        if "Subpopulations" not in sv_file:
            sv_metric_name = sv_file.split("_")[1]

            if sv_file == "SV_F1Score":
                # For now, this one will not be used
                grouped_sv = results_dataframe.groupby(["Evaluator", "Classes"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Classes", "Round"], axis=1)

            elif sv_file == "SV_CosineSimilarity":
                grouped_sv = results_dataframe.groupby(["Evaluator"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Round"], axis=1)
                sv_metric.append(sv_metric_name)
                sv_metric_c_a.append("C")
                for client in clients_sv.keys():
                    clients_sv[client].append(-1 * grouped_sv.loc["Centralized", str(client)])
            else:
                grouped_sv = results_dataframe.groupby(["Evaluator"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Round"], axis=1)
                sv_metric.append(sv_metric_name)
                sv_metric.append(sv_metric_name)
                sv_metric_c_a.append("C")
                sv_metric_c_a.append("A")
                for client in clients_sv.keys():
                    # The values are multiplied for -1 to prepare the data for a posterior sort.
                    try:
                        clients_sv[client].append(-1 * grouped_sv.loc["Centralized", str(client)])
                        clients_sv[client].append(-1 * grouped_sv.loc["Aggregated", str(client)])
                    except KeyError:
                        print(f"Aggregation method: {aggregation_method}")
                        print(f"Dataset: {dataset_name}")
                        print(f"Partition: {additional_parameter}")
                        print(f"Random seed: {random_seed}")
                        print(f"SV metric: {sv_metric}")

    maverick_ranked["Aggregation_Process"] = [aggregation_method for _ in range(len(sv_metric))]
    maverick_ranked["Dataset"] = [dataset_name for _ in range(len(sv_metric))]
    maverick_ranked["Partition"] = [additional_parameter for _ in range(len(sv_metric))]
    maverick_ranked["RandomSeed"] = [random_seed for _ in range(len(sv_metric))]
    maverick_ranked["Metric"] = sv_metric
    maverick_ranked["C/A"] = sv_metric_c_a

    for client in clients_sv.keys():
        maverick_ranked[client] = clients_sv[client]

    for index, row in maverick_ranked.iterrows():
        sorted_clients = np.argsort(maverick_ranked.loc[index, [k for k in range(num_clients)]])
        for client, position in zip(sorted_clients, range(1, 7)):
            maverick_ranked.loc[index, client] = position

    return maverick_ranked

In [100]:
maverick_ranked = pd.DataFrame()

training_configurations_mavericks = [scenario for scenario in training_configurations_for_distances
                                    if scenario["partition"] == "manual" and scenario.get("is_maverick_scenario", False) == True and scenario.get("exclude_maverick", False) == False]

for training_configuration in training_configurations_mavericks:
    aggregation_process = training_configuration["strategy"]
    dataset_name = training_configuration["dataset"]
    type_of_partition = training_configuration["partition"]
    additional_parameter = training_configuration["name"]
    # additional_parameter = configuration["Dirichlet/NamePartition"]
    random_seed = training_configuration["run_number"]
    # exclude_maverick = training_configuration["exclude_maverick"]
    # maverick_number = training_configuration["maverick_client_number"]
    # dataset_name, type_of_partition, additional_parameter, group_partitions, num_clients, maverick
    maverick_ranked = pd.concat([maverick_ranked, ranking_Mavericks(aggregation_process, dataset_name, type_of_partition, additional_parameter)], ignore_index=True)

In [101]:
# display(maverick_ranked[(maverick_ranked["C/A"] == "C")].groupby(["Dataset", "Partition", "Metric"]).mean(numeric_only=True))
# display(maverick_ranked[(maverick_ranked["C/A"] == "C")].groupby(["Dataset", "Partition", "Metric"]).std(numeric_only=True))
#
display(maverick_ranked)
display(maverick_ranked[(maverick_ranked["C/A"] == "C")].groupby(["Aggregation_Process", "Dataset", "Partition"]).mean(numeric_only=True))
display(maverick_ranked[(maverick_ranked["C/A"] == "C")].groupby(["Aggregation_Process", "Dataset", "Partition"]).std(numeric_only=True))

display(maverick_ranked[(maverick_ranked["C/A"] == "C")].groupby(["Dataset", "Partition", "Metric"]).mean(numeric_only=True))
display(maverick_ranked[(maverick_ranked["C/A"] == "C")].groupby(["Dataset", "Partition", "Metric"]).std(numeric_only=True))

# display(maverick_ranked[(maverick_ranked["C/A"] == "C") & (maverick_ranked["Aggregation_Process"] == "FedProx")].groupby(["Dataset", "Partition", "Metric"]).mean(numeric_only=True))
# display(maverick_ranked[(maverick_ranked["C/A"] == "C") & (maverick_ranked["Aggregation_Process"] == "FedProx")].groupby(["Dataset", "Partition", "Metric"]).std(numeric_only=True))
#
# display(maverick_ranked[(maverick_ranked["C/A"] == "C") & (maverick_ranked["Aggregation_Process"] == "Scaffold")].groupby(["Dataset", "Partition", "Metric"]).mean(numeric_only=True))
# display(maverick_ranked[(maverick_ranked["C/A"] == "C") & (maverick_ranked["Aggregation_Process"] == "Scaffold")].groupby(["Dataset", "Partition", "Metric"]).std(numeric_only=True))
#
# display(maverick_ranked[maverick_ranked["Metric"] != "CosineSimilarity"].groupby(["C/A"]).mean(numeric_only=True))
# display(maverick_ranked[maverick_ranked["Metric"] != "CosineSimilarity"].groupby(["C/A"]).std(numeric_only=True))

,Aggregation_Process,Dataset,Partition,RandomSeed,Metric,C/A,0,1,2,3,4,5
0,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,1,MAE,C,5.0,3.0,6.0,4.0,1.0,2.0
1,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,1,MAE,A,5.0,3.0,6.0,4.0,1.0,2.0
2,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,1,CosineSimilarity,C,3.0,1.0,6.0,4.0,5.0,2.0
3,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,1,MSE,C,5.0,4.0,6.0,2.0,1.0,3.0
4,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,1,MSE,A,5.0,4.0,6.0,2.0,1.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1525,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_ddos_udp,5,F1ScoreMacro,C,2.0,3.0,4.0,5.0,1.0,NaN
1526,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_ddos_udp,5,F1ScoreMacro,A,2.0,3.0,4.0,5.0,1.0,NaN
1527,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_ddos_udp,5,Accuracy,C,3.0,2.0,4.0,5.0,1.0,NaN
1528,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_ddos_udp,5,Accuracy,A,3.0,2.0,4.0,5.0,1.0,NaN


RandomSeed  \
Aggregation_Process Dataset          Partition                                                        
FedAvg              edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                     3.0   
                                     edgeiot_coreset_1_Maverick_Only_Normal                     3.0   
                                     edgeiot_coreset_1_Maverick_ddos_udp                        3.0   
                                     edgeiot_coreset_1_Maverick_sql_injection                   3.0   
                    har              HAR_1_Maverick_Laying                                      3.0   
                                     HAR_1_Maverick_Laying_Balanced                             3.0   
                                     HAR_1_Maverick_WalkingUpstairs                             3.0   
                                     HAR_1_Maverick_WalkingUpstairs_Balanced                    3.0   
                    wids_dataset     wids_energy_maverick_facility_type_grocery_store           3.0   
                                     wids_energy_maverick_facility_type_uncategorize...         3.0   
FedProx             edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                     3.0   
                                     edgeiot_coreset_1_Maverick_Only_Normal                     3.0   
                                     edgeiot_coreset_1_Maverick_ddos_udp                        3.0   
                                     edgeiot_coreset_1_Maverick_sql_injection                   3.0   
                    har              HAR_1_Maverick_Laying                                      3.0   
                                     HAR_1_Maverick_Laying_Balanced                             3.0   
                                     HAR_1_Maverick_WalkingUpstairs                             3.0   
                                     HAR_1_Maverick_WalkingUpstairs_Balanced                    3.0   
                    wids_dataset     wids_energy_maverick_facility_type_grocery_store           3.0   
                                     wids_energy_maverick_facility_type_uncategorize...         3.0   
Scaffold            edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                     3.0   
                                     edgeiot_coreset_1_Maverick_Only_Normal                     3.0   
                                     edgeiot_coreset_1_Maverick_ddos_udp                        3.0   
                                     edgeiot_coreset_1_Maverick_sql_injection                   3.0   
                    har              HAR_1_Maverick_Laying                                      3.0   
                                     HAR_1_Maverick_Laying_Balanced                             3.0   
                                     HAR_1_Maverick_WalkingUpstairs                             3.0   
                                     HAR_1_Maverick_WalkingUpstairs_Balanced                    3.0   
                    wids_dataset     wids_energy_maverick_facility_type_grocery_store           3.0   
                                     wids_energy_maverick_facility_type_uncategorize...         3.0   

                                                                                                0  \
Aggregation_Process Dataset          Partition                                                      
FedAvg              edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class              2.966667   
                                     edgeiot_coreset_1_Maverick_Only_Normal              3.433333   
                                     edgeiot_coreset_1_Maverick_ddos_udp                 3.066667   
                                     edgeiot_coreset_1_Maverick_sql_injection            2.333333   
                    har              HAR_1_Maverick_Laying                               1.000000   
                                     HAR_1_Maverick_Laying_Balanced                  

RandomSeed  \
Aggregation_Process Dataset          Partition                                                        
FedAvg              edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                1.438390   
                                     edgeiot_coreset_1_Maverick_Only_Normal                1.438390   
                                     edgeiot_coreset_1_Maverick_ddos_udp                   1.438390   
                                     edgeiot_coreset_1_Maverick_sql_injection              1.438390   
                    har              HAR_1_Maverick_Laying                                 1.438390   
                                     HAR_1_Maverick_Laying_Balanced                        1.438390   
                                     HAR_1_Maverick_WalkingUpstairs                        1.438390   
                                     HAR_1_Maverick_WalkingUpstairs_Balanced               1.438390   
                    wids_dataset     wids_energy_maverick_facility_type_grocery_store      1.450953   
                                     wids_energy_maverick_facility_type_uncategorize...    1.450953   
FedProx             edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                1.438390   
                                     edgeiot_coreset_1_Maverick_Only_Normal                1.438390   
                                     edgeiot_coreset_1_Maverick_ddos_udp                   1.438390   
                                     edgeiot_coreset_1_Maverick_sql_injection              1.438390   
                    har              HAR_1_Maverick_Laying                                 1.438390   
                                     HAR_1_Maverick_Laying_Balanced                        1.438390   
                                     HAR_1_Maverick_WalkingUpstairs                        1.438390   
                                     HAR_1_Maverick_WalkingUpstairs_Balanced               1.438390   
                    wids_dataset     wids_energy_maverick_facility_type_grocery_store      1.450953   
                                     wids_energy_maverick_facility_type_uncategorize...    1.450953   
Scaffold            edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                1.438390   
                                     edgeiot_coreset_1_Maverick_Only_Normal                1.438390   
                                     edgeiot_coreset_1_Maverick_ddos_udp                   1.438390   
                                     edgeiot_coreset_1_Maverick_sql_injection              1.438390   
                    har              HAR_1_Maverick_Laying                                 1.438390   
                                     HAR_1_Maverick_Laying_Balanced                        1.438390   
                                     HAR_1_Maverick_WalkingUpstairs                        1.438390   
                                     HAR_1_Maverick_WalkingUpstairs_Balanced               1.438390   
                    wids_dataset     wids_energy_maverick_facility_type_grocery_store      1.450953   
                                     wids_energy_maverick_facility_type_uncategorize...    1.450953   

                                                                                                0  \
Aggregation_Process Dataset          Partition                                                      
FedAvg              edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class              1.245221   
                                     edgeiot_coreset_1_Maverick_Only_Normal              0.971431   
                                     edgeiot_coreset_1_Maverick_ddos_udp                 1.112107   
                                     edgeiot_coreset_1_Maverick_sql_injection            0.844182   
                    har              HAR_1_Maverick_Laying                               0.000000   
                                     HAR_1_Maverick_Laying_Balanced                  

RandomSeed  \
Dataset          Partition                                          Metric                         
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class             Accuracy                 3.0   
                                                                    CosineSimilarity         3.0   
                                                                    CrossEntropyLoss         3.0   
                                                                    F1ScoreMacro             3.0   
                                                                    F1ScoreMicro             3.0   
                                                                    MCC                      3.0   
                 edgeiot_coreset_1_Maverick_Only_Normal             Accuracy                 3.0   
                                                                    CosineSimilarity         3.0   
                                                                    CrossEntropyLoss         3.0   
                                                                    F1ScoreMacro             3.0   
                                                                    F1ScoreMicro             3.0   
                                                                    MCC                      3.0   
                 edgeiot_coreset_1_Maverick_ddos_udp                Accuracy                 3.0   
                                                                    CosineSimilarity         3.0   
                                                                    CrossEntropyLoss         3.0   
                                                                    F1ScoreMacro             3.0   
                                                                    F1ScoreMicro             3.0   
                                                                    MCC                      3.0   
                 edgeiot_coreset_1_Maverick_sql_injection           Accuracy                 3.0   
                                                                    CosineSimilarity         3.0   
                                                                    CrossEntropyLoss         3.0   
                                                                    F1ScoreMacro             3.0   
                                                                    F1ScoreMicro             3.0   
                                                                    MCC                      3.0   
har              HAR_1_Maverick_Laying                              Accuracy                 3.0   
                                                                    CosineSimilarity         3.0   
                                                                    CrossEntropyLoss         3.0   
                                                                    F1ScoreMacro             3.0   
                                                                    F1ScoreMicro             3.0   
                                                                    MCC                      3.0   
                 HAR_1_Maverick_Laying_Balanced                     Accuracy                 3.0   
                                                                    CosineSimilarity         3.0   
                                                                    CrossEntropyLoss         3.0   
                                                                    F1ScoreMacro             3.0   
                                                                    F1ScoreMicro             3.0   
                                                                    MCC                      3.0   
                 HAR_1_Maverick_WalkingUpstairs                     Accuracy                 3.0   
                                                                    CosineSimilarity         3.0   
                                                                    CrossEntropyLoss  

RandomSeed  \
Dataset          Partition                                          Metric                         
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class             Accuracy             1.46385   
                                                                    CosineSimilarity     1.46385   
                                                                    CrossEntropyLoss     1.46385   
                                                                    F1ScoreMacro         1.46385   
                                                                    F1ScoreMicro         1.46385   
                                                                    MCC                  1.46385   
                 edgeiot_coreset_1_Maverick_Only_Normal             Accuracy             1.46385   
                                                                    CosineSimilarity     1.46385   
                                                                    CrossEntropyLoss     1.46385   
                                                                    F1ScoreMacro         1.46385   
                                                                    F1ScoreMicro         1.46385   
                                                                    MCC                  1.46385   
                 edgeiot_coreset_1_Maverick_ddos_udp                Accuracy             1.46385   
                                                                    CosineSimilarity     1.46385   
                                                                    CrossEntropyLoss     1.46385   
                                                                    F1ScoreMacro         1.46385   
                                                                    F1ScoreMicro         1.46385   
                                                                    MCC                  1.46385   
                 edgeiot_coreset_1_Maverick_sql_injection           Accuracy             1.46385   
                                                                    CosineSimilarity     1.46385   
                                                                    CrossEntropyLoss     1.46385   
                                                                    F1ScoreMacro         1.46385   
                                                                    F1ScoreMicro         1.46385   
                                                                    MCC                  1.46385   
har              HAR_1_Maverick_Laying                              Accuracy             1.46385   
                                                                    CosineSimilarity     1.46385   
                                                                    CrossEntropyLoss     1.46385   
                                                                    F1ScoreMacro         1.46385   
                                                                    F1ScoreMicro         1.46385   
                                                                    MCC                  1.46385   
                 HAR_1_Maverick_Laying_Balanced                     Accuracy             1.46385   
                                                                    CosineSimilarity     1.46385   
                                                                    CrossEntropyLoss     1.46385   
                                                                    F1ScoreMacro         1.46385   
                                                                    F1ScoreMicro         1.46385   
                                                                    MCC                  1.46385   
                 HAR_1_Maverick_WalkingUpstairs                     Accuracy             1.46385   
                                                                    CosineSimilarity     1.46385   
                                                                    CrossEntropyLoss  

In [102]:
display(maverick_ranked[(maverick_ranked["C/A"] == "C")].groupby(["Partition", "Metric"]).mean(numeric_only=True))

RandomSeed  \
Partition                                          Metric                         
HAR_1_Maverick_Laying                              Accuracy                 3.0   
                                                   CosineSimilarity         3.0   
                                                   CrossEntropyLoss         3.0   
                                                   F1ScoreMacro             3.0   
                                                   F1ScoreMicro             3.0   
                                                   MCC                      3.0   
HAR_1_Maverick_Laying_Balanced                     Accuracy                 3.0   
                                                   CosineSimilarity         3.0   
                                                   CrossEntropyLoss         3.0   
                                                   F1ScoreMacro             3.0   
                                                   F1ScoreMicro             3.0   
                                                   MCC                      3.0   
HAR_1_Maverick_WalkingUpstairs                     Accuracy                 3.0   
                                                   CosineSimilarity         3.0   
                                                   CrossEntropyLoss         3.0   
                                                   F1ScoreMacro             3.0   
                                                   F1ScoreMicro             3.0   
                                                   MCC                      3.0   
HAR_1_Maverick_WalkingUpstairs_Balanced            Accuracy                 3.0   
                                                   CosineSimilarity         3.0   
                                                   CrossEntropyLoss         3.0   
                                                   F1ScoreMacro             3.0   
                                                   F1ScoreMicro             3.0   
                                                   MCC                      3.0   
edgeiot_coreset_1_Maverick_Least_Class             Accuracy                 3.0   
                                                   CosineSimilarity         3.0   
                                                   CrossEntropyLoss         3.0   
                                                   F1ScoreMacro             3.0   
                                                   F1ScoreMicro             3.0   
                                                   MCC                      3.0   
edgeiot_coreset_1_Maverick_Only_Normal             Accuracy                 3.0   
                                                   CosineSimilarity         3.0   
                                                   CrossEntropyLoss         3.0   
                                                   F1ScoreMacro             3.0   
                                                   F1ScoreMicro             3.0   
                                                   MCC                      3.0   
edgeiot_coreset_1_Maverick_ddos_udp                Accuracy                 3.0   
                                                   CosineSimilarity         3.0   
                                                   CrossEntropyLoss         3.0   
                                                   F1ScoreMacro             3.0   
                                                   F1ScoreMicro             3.0   
                                                   MCC                      3.0   
edgeiot_coreset_1_Maverick_sql_injection           Accuracy                 3.0   
                                                   CosineSimilarity         3.0   
                                                   CrossEntropyLoss         3.0   
                                                   F1ScoreMacro             3.0   
                                                   F1ScoreMicro             3.0   
  

In [103]:
def maverick_vs_distance(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick, type_of_loss):
    value_distance_dataframe = pd.DataFrame()

    path_to_result_dataframes = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True, False)
    path_to_distance_dataframes = get_distances_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter)

    sv_value_centralized = []
    clients_sv = {k: [] for k in range((len(os.listdir(path_to_result_dataframes))/4)-1)}
    sv_metric = []

    for sv_file in os.listdir(path_to_result_dataframes + os.sep + "Shapley_Value"):
        results_dataframe = pd.read_csv(path_to_result_dataframes + os.sep + "Shapley_Value" + os.sep + sv_file)
        sv_metric_name = sv_file.split("_")[1]
        if sv_file == "SV_CosineSimilarity":
            grouped_sv = results_dataframe.groupby(["Evaluator"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Round"], axis=1)
            sv_metric.append(sv_metric_name)
            # sv_value_centralized.append(grouped_sv.loc["Centralized"])
            for client in clients_sv.keys():
                clients_sv[client].append(grouped_sv.loc["Centralized", str(client)])
        elif sv_file == "SV_CrossEntropyLoss":
            grouped_sv = results_dataframe.groupby(["Evaluator"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Round"], axis=1)
            sv_metric.append(sv_metric_name)
            for client in clients_sv.keys():
                clients_sv[client].append(grouped_sv.loc["Centralized", str(client)])
        elif sv_file == "SV_RMSE":
            grouped_sv = results_dataframe.groupby(["Evaluator"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Round"], axis=1)
            sv_metric.append(sv_metric_name)
            for client in clients_sv.keys():
                clients_sv[client].append(grouped_sv.loc["Centralized", str(client)])
    
    for distance_file in os.listdir(path_to_distance_dataframes):
        distance_dataframe = pd.read_csv(path_to_distance_dataframes + os.sep + distance_file, index_col=0)
        distance_metric_name = distance_file.split(".")[0]
        sv_metric.append(distance_metric_name)
        
        if distance_file == "volume.csv":
            for client in clients_sv.keys():
                clients_sv[client].append(distance_dataframe.loc[str(client)].iloc[0])
        
        else:
            for client in clients_sv.keys():
                clients_sv[client].append(distance_dataframe.T.loc["Global", client])

    value_distance_dataframe["AggregationProcess"] = [aggregation_process for _ in range(len(sv_metric))]
    value_distance_dataframe["Dataset"] = [dataset_name for _ in range(len(sv_metric))]
    value_distance_dataframe["Partition"] = [additional_parameter for _ in range(len(sv_metric))]
    value_distance_dataframe["Metric"] = sv_metric
    value_distance_dataframe["Maverick"] = maverick
    for client in clients_sv.keys():
        value_distance_dataframe[str(client)] = clients_sv[client]

    return value_distance_dataframe

In [104]:
def ranking_valuation_method_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick, type_of_loss):
    value_distance_dataframe = pd.DataFrame()

    path_to_result_dataframes = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True)
    number_of_metrics = 0

    sv_metric = []
    # sv_metric_theoretical_correct_rank = []
    # sv_metric_correct_rank = []
    sv_correct_rank = []

    for file_path in os.listdir(path_to_result_dataframes + os.sep + "Shapley_Value"):
        if "Subpopulation" not in file_path:
            number_of_metrics += 1
            results_dataframe = pd.read_csv(path_to_result_dataframes + os.sep + "Shapley_Value" + os.sep + file_path, index_col=0)
            grouped_sv = results_dataframe.groupby(["Evaluator"]).sum(numeric_only=True).reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator"], axis=1)
            only_centralized_results = grouped_sv.loc["Centralized"]
            best_valued_index = int(only_centralized_results[only_centralized_results == max(only_centralized_results)].index[0])

            valuation_method_name = file_path.split(".")[0]
            sv_metric.append(valuation_method_name)

            if best_valued_index == maverick:
                sv_correct_rank.append(1)
            else:
                sv_correct_rank.append(0)

    value_distance_dataframe["AggregationProcess"] = [aggregation_process for _ in range(len(sv_metric))]
    value_distance_dataframe["Dataset"] = [dataset_name for _ in range(len(sv_metric))]
    value_distance_dataframe["Partition"] = [additional_parameter for _ in range(len(sv_metric))]
    value_distance_dataframe["SV_Metric"] = sv_metric
    value_distance_dataframe["Maverick"] = maverick
    value_distance_dataframe["Correct_SV"] = sv_correct_rank

    return value_distance_dataframe

In [105]:
distance_and_rank = pd.DataFrame()

training_configurations_mavericks = [scenario for scenario in training_configurations_for_distances
                                    if scenario["partition"] == "manual" and scenario.get("is_maverick_scenario", False) == True and scenario.get("exclude_maverick", False) == False]

for training_configuration in training_configurations_mavericks:
    aggregation_process = training_configuration["strategy"]
    dataset_name = training_configuration["dataset"]
    type_of_partition = training_configuration["partition"]
    additional_parameter = training_configuration["name"]
    # additional_parameter = configuration["Dirichlet/NamePartition"]
    random_seed = training_configuration["run_number"]
    maverick = training_configuration["maverick_client_number"]
    if dataset_name == "wids_dataset":
        type_of_loss = "MSE"
    else:
        type_of_loss = "CrossEntropyLoss"

    distance_and_rank = pd.concat([distance_and_rank, ranking_valuation_method_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick, type_of_loss)], ignore_index=True)

In [106]:
grouped_distance_and_rank = distance_and_rank[distance_and_rank["Dataset"] == "wids_dataset"]
display(grouped_distance_and_rank)

,AggregationProcess,Dataset,Partition,SV_Metric,Maverick,Correct_SV
0,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,SV_MAE,5,0
1,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,SV_CosineSimilarity,5,0
2,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,SV_MSE,5,0
3,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,SV_RMSE,5,0
4,FedAvg,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,SV_MAE,5,0
...,...,...,...,...,...,...
675,Scaffold,wids_dataset,wids_energy_maverick_facility_type_grocery_store,SV_RMSE,5,0
676,Scaffold,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,SV_MAE,5,0
677,Scaffold,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,SV_CosineSimilarity,5,0
678,Scaffold,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,SV_MSE,5,0


In [107]:
grouped_distance_and_rank = distance_and_rank.groupby("SV_Metric").agg({'Maverick':'count', 'Correct_SV': 'sum'}).T

In [108]:
grouped_distance_and_rank

SV_Metric,SV_Accuracy,SV_CosineSimilarity,SV_CrossEntropyLoss,SV_F1Score,SV_F1ScoreMacro,SV_F1ScoreMicro,SV_MAE,SV_MCC,SV_MSE,SV_RMSE
Maverick,120,150,120,120,120,120,30,120,30,30
Correct_SV,71,40,113,106,106,69,6,72,8,7


In [109]:
# grouped_distance_and_rank = distance_and_rank[distance_and_rank["Dataset"] == "wids_dataset"].groupby("Distance").agg({'Maverick':'count', 'Correct_SV': 'sum'}).T

KeyError: 'Distance'

In [110]:
def ranking_distance_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick, type_of_loss):
    value_distance_dataframe = pd.DataFrame()

    path_to_result_dataframes = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True)

    if type_of_loss == "MSE":
        results_dataframe = pd.read_csv(path_to_result_dataframes + os.sep + "Shapley_Value" + os.sep + "SV_MSE")
    else:
        results_dataframe = pd.read_csv(path_to_result_dataframes + os.sep + "Shapley_Value" + os.sep + "SV_CrossEntropyLoss")

    grouped_sv = results_dataframe.groupby(["Evaluator"]).sum().reindex(sorted(results_dataframe.columns), axis=1).drop(["Evaluator", "Round"], axis=1)

    only_centralized_results = grouped_sv.loc["Centralized"]
    best_valued_index = int(only_centralized_results[only_centralized_results == max(only_centralized_results)].index[0])

    path_to_distance_dataframes = get_distances_from_route(dataset_name, type_of_partition, additional_parameter, random_seed)

    distance_dictionary = {
        "performanceDegradation": "I",
        "yShiftDataframe": "I",
        "relevance": "D",
        "diversity": "D",
        "hellinger": "I",
        "wasserstein": "I",
        "gaussian_mmd": "I",
        "negativeConditionalEntropy": "D",
        "volume": "D"
    }

    distance = []
    distance_theoretical_correct_rank = []
    distance_sv_correct_rank = []
    sv_correct_rank = []

    # if best_valued_index == maverick:
    #     sv_correct_rank.append(1)
    # else:
    #     sv_correct_rank.append(0)
    
    for distance_file in os.listdir(path_to_distance_dataframes):
        distance_dataframe = pd.read_csv(path_to_distance_dataframes + os.sep + distance_file, index_col=0)
        distance_metric_name = distance_file.split(".")[0]
        distance.append(distance_metric_name)
        
        if distance_file == "volume.csv":
            volumes = distance_dataframe.iloc[:, 0]
            # volumes.pop("Global")
            volumes.reset_index(inplace=True, drop=True)
            if volumes[volumes == max(volumes)].index[0] == maverick:
                distance_theoretical_correct_rank.append(1)
            else:
                distance_theoretical_correct_rank.append(0)
            
            if volumes[volumes == max(volumes)].index[0] == best_valued_index:
                distance_sv_correct_rank.append(1)
            else:
                distance_sv_correct_rank.append(0)
        
        else:
            distance_values = distance_dataframe.loc[:, "Global"]
            if distance_dictionary[distance_metric_name] == "D":
                best_distance_index = int(distance_values[distance_values == max(distance_values)].index[0])
                if best_distance_index == maverick:
                    distance_theoretical_correct_rank.append(1)
                else:
                    distance_theoretical_correct_rank.append(0)

                if best_distance_index == best_valued_index:
                    distance_sv_correct_rank.append(1)
                else:
                    distance_sv_correct_rank.append(0)
            
            else:
                best_distance_index = int(distance_values[distance_values == min(distance_values)].index[0])
                if best_distance_index == maverick:
                    distance_theoretical_correct_rank.append(1)
                else:
                    distance_theoretical_correct_rank.append(0)
                
                if best_distance_index == best_valued_index:
                    distance_sv_correct_rank.append(1)
                else:
                    distance_sv_correct_rank.append(0)

    value_distance_dataframe["AggregationProcess"] = [aggregation_process for _ in range(len(distance))]
    value_distance_dataframe["Dataset"] = [dataset_name for _ in range(len(distance))]
    value_distance_dataframe["Partition"] = [additional_parameter for _ in range(len(distance))]
    value_distance_dataframe["Distance"] = distance
    value_distance_dataframe["Maverick"] = maverick
    value_distance_dataframe["Correct_Theoretical"] = distance_theoretical_correct_rank
    value_distance_dataframe["Correct_SV"] = distance_sv_correct_rank

    return value_distance_dataframe

In [111]:
distance_and_rank = pd.DataFrame()

training_configurations_mavericks = [scenario for scenario in training_configurations_for_distances
                                    if scenario["partition"] == "manual" and scenario.get("is_maverick_scenario", False) == True and scenario.get("exclude_maverick", False) == False]

for training_configuration in training_configurations_mavericks:
    aggregation_process = training_configuration["strategy"]
    dataset_name = training_configuration["dataset"]
    type_of_partition = training_configuration["partition"]
    additional_parameter = training_configuration["name"]
    # additional_parameter = configuration["Dirichlet/NamePartition"]
    random_seed = training_configuration["run_number"]
    maverick = training_configuration["maverick_client_number"]
    if dataset_name == "wids_dataset":
        type_of_loss = "MSE"
    else:
        type_of_loss = "CrossEntropyLoss"

    distance_and_rank = pd.concat([distance_and_rank, ranking_distance_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick, type_of_loss)], ignore_index=True)

In [112]:
display(distance_and_rank[distance_and_rank["Distance"] == "gaussian_mmd"])
display(distance_and_rank[distance_and_rank["Distance"] == "wasserstein"])
display(distance_and_rank[distance_and_rank["Distance"] == "hellinger"])
display(distance_and_rank[distance_and_rank["Distance"] == "volume"])

,AggregationProcess,Dataset,Partition,Distance,Maverick,Correct_Theoretical,Correct_SV
5,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,gaussian_mmd,5,0,0
12,FedAvg,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,gaussian_mmd,5,0,0
19,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,gaussian_mmd,5,1,0
26,FedAvg,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,gaussian_mmd,5,1,0
33,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,gaussian_mmd,5,0,0
...,...,...,...,...,...,...,...
1136,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_ddos_udp,gaussian_mmd,4,0,0
1144,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Least_Class,gaussian_mmd,4,0,0
1152,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,gaussian_mmd,4,1,1
1160,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_sql_injection,gaussian_mmd,4,0,0


,AggregationProcess,Dataset,Partition,Distance,Maverick,Correct_Theoretical,Correct_SV
4,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,wasserstein,5,0,0
11,FedAvg,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,wasserstein,5,1,0
18,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,wasserstein,5,0,0
25,FedAvg,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,wasserstein,5,0,0
32,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,wasserstein,5,0,0
...,...,...,...,...,...,...,...
1135,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_ddos_udp,wasserstein,4,0,0
1143,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Least_Class,wasserstein,4,0,0
1151,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,wasserstein,4,1,1
1159,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_sql_injection,wasserstein,4,0,0


,AggregationProcess,Dataset,Partition,Distance,Maverick,Correct_Theoretical,Correct_SV
2,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,hellinger,5,1,0
9,FedAvg,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,hellinger,5,1,0
16,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,hellinger,5,0,0
23,FedAvg,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,hellinger,5,0,0
30,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,hellinger,5,0,0
...,...,...,...,...,...,...,...
1132,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_ddos_udp,hellinger,4,0,0
1140,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Least_Class,hellinger,4,0,0
1148,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,hellinger,4,1,1
1156,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_sql_injection,hellinger,4,0,0


,AggregationProcess,Dataset,Partition,Distance,Maverick,Correct_Theoretical,Correct_SV
6,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,volume,5,1,0
13,FedAvg,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,volume,5,1,0
20,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,volume,5,0,0
27,FedAvg,wids_dataset,wids_energy_maverick_facility_type_uncategoriz...,volume,5,0,0
34,FedAvg,wids_dataset,wids_energy_maverick_facility_type_grocery_store,volume,5,0,0
...,...,...,...,...,...,...,...
1137,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_ddos_udp,volume,4,0,0
1145,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Least_Class,volume,4,1,1
1153,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,volume,4,1,1
1161,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_sql_injection,volume,4,1,1


In [113]:
display(distance_and_rank[(distance_and_rank["Dataset"] == "har") & (distance_and_rank["Distance"] == "volume")])
display(distance_and_rank[(distance_and_rank["Dataset"] == "har") & (distance_and_rank["Distance"] == "wasserstein")])
display(distance_and_rank[(distance_and_rank["Dataset"] == "har") & (distance_and_rank["Distance"] == "hellinger")])
display(distance_and_rank[(distance_and_rank["Dataset"] == "har") & (distance_and_rank["Distance"] == "gaussian_mmd")])

,AggregationProcess,Dataset,Partition,Distance,Maverick,Correct_Theoretical,Correct_SV
77,FedAvg,har,HAR_1_Maverick_Laying,volume,0,1,1
85,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,volume,5,1,1
93,FedAvg,har,HAR_1_Maverick_Laying_Balanced,volume,0,1,1
101,FedAvg,har,HAR_1_Maverick_WalkingUpstairs_Balanced,volume,5,1,1
109,FedAvg,har,HAR_1_Maverick_Laying,volume,0,1,1
117,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,volume,5,1,1
125,FedAvg,har,HAR_1_Maverick_Laying_Balanced,volume,0,1,1
133,FedAvg,har,HAR_1_Maverick_WalkingUpstairs_Balanced,volume,5,1,1
141,FedAvg,har,HAR_1_Maverick_Laying,volume,0,1,1
149,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,volume,5,1,1


,AggregationProcess,Dataset,Partition,Distance,Maverick,Correct_Theoretical,Correct_SV
75,FedAvg,har,HAR_1_Maverick_Laying,wasserstein,0,0,0
83,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,wasserstein,5,0,0
91,FedAvg,har,HAR_1_Maverick_Laying_Balanced,wasserstein,0,1,1
99,FedAvg,har,HAR_1_Maverick_WalkingUpstairs_Balanced,wasserstein,5,1,1
107,FedAvg,har,HAR_1_Maverick_Laying,wasserstein,0,0,0
115,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,wasserstein,5,0,0
123,FedAvg,har,HAR_1_Maverick_Laying_Balanced,wasserstein,0,0,0
131,FedAvg,har,HAR_1_Maverick_WalkingUpstairs_Balanced,wasserstein,5,0,0
139,FedAvg,har,HAR_1_Maverick_Laying,wasserstein,0,0,0
147,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,wasserstein,5,0,0


,AggregationProcess,Dataset,Partition,Distance,Maverick,Correct_Theoretical,Correct_SV
72,FedAvg,har,HAR_1_Maverick_Laying,hellinger,0,1,1
80,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,hellinger,5,0,0
88,FedAvg,har,HAR_1_Maverick_Laying_Balanced,hellinger,0,1,1
96,FedAvg,har,HAR_1_Maverick_WalkingUpstairs_Balanced,hellinger,5,1,1
104,FedAvg,har,HAR_1_Maverick_Laying,hellinger,0,1,1
112,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,hellinger,5,0,0
120,FedAvg,har,HAR_1_Maverick_Laying_Balanced,hellinger,0,1,1
128,FedAvg,har,HAR_1_Maverick_WalkingUpstairs_Balanced,hellinger,5,1,1
136,FedAvg,har,HAR_1_Maverick_Laying,hellinger,0,1,1
144,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,hellinger,5,0,0


,AggregationProcess,Dataset,Partition,Distance,Maverick,Correct_Theoretical,Correct_SV
76,FedAvg,har,HAR_1_Maverick_Laying,gaussian_mmd,0,1,1
84,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,gaussian_mmd,5,1,1
92,FedAvg,har,HAR_1_Maverick_Laying_Balanced,gaussian_mmd,0,1,1
100,FedAvg,har,HAR_1_Maverick_WalkingUpstairs_Balanced,gaussian_mmd,5,1,1
108,FedAvg,har,HAR_1_Maverick_Laying,gaussian_mmd,0,1,1
116,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,gaussian_mmd,5,1,1
124,FedAvg,har,HAR_1_Maverick_Laying_Balanced,gaussian_mmd,0,1,1
132,FedAvg,har,HAR_1_Maverick_WalkingUpstairs_Balanced,gaussian_mmd,5,1,1
140,FedAvg,har,HAR_1_Maverick_Laying,gaussian_mmd,0,1,1
148,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,gaussian_mmd,5,1,1


In [ ]:
display(distance_and_rank[(distance_and_rank["Distance"] == "volume") & (distance_and_rank["AggregationProcess"] == "FedAvg")])

In [ ]:
results_dataframe = pd.DataFrame()

results_dataframe["Number_Classification_Scenarios"] = distance_and_rank[distance_and_rank["Dataset"] != "wids_dataset"].groupby(["Distance"])["Correct_Theoretical"].count()
results_dataframe["Number_Regression_Scenarios"] = distance_and_rank[distance_and_rank["Dataset"] == "wids_dataset"].groupby(["Distance"])["Correct_Theoretical"].count()

correct_regression = distance_and_rank[distance_and_rank["Dataset"] == "wids_dataset"].groupby(["Distance"])[["Correct_Theoretical", "Correct_SV"]].sum(numeric_only=True)
correct_regression.rename({"Correct_Theoretical": "Maverick_Theory_regression", "Correct_SV": "Matches_Most_Value_SV_regression"}, axis=1, inplace=True)
correct_classification = distance_and_rank[distance_and_rank["Dataset"] != "wids_dataset"].groupby(["Distance"])[["Correct_Theoretical", "Correct_SV"]].sum(numeric_only=True)
correct_classification.rename({"Correct_Theoretical": "Maverick_Theory_classification", "Correct_SV": "Matches_Most_Value_SV_classification"}, axis=1, inplace=True)

results_dataframe = pd.concat([results_dataframe, correct_classification, correct_regression], axis=1)
#
results_dataframe["Accuracy_Classification_Theory"] = results_dataframe["Maverick_Theory_classification"] / results_dataframe["Number_Classification_Scenarios"]
results_dataframe["Accuracy_Classification_SV"] = results_dataframe["Matches_Most_Value_SV_classification"] / results_dataframe["Number_Classification_Scenarios"]

results_dataframe["Accuracy_Regression_Theory"] = results_dataframe["Maverick_Theory_regression"] / results_dataframe["Number_Regression_Scenarios"]
results_dataframe["Accuracy_Regression_SV"] = results_dataframe["Matches_Most_Value_SV_regression"] / results_dataframe["Number_Regression_Scenarios"]

display(results_dataframe)

##### Checking value for Maverick on unique data

Checking value per Label

In [114]:
def f1_score_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick_number, maverick_label):
    value_distance_dataframe = pd.DataFrame()

    path_to_result_dataframes_with_maverick = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True, False)
    path_to_result_dataframes_without_maverick = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True, True)

    f1_score_with_maverick = pd.read_csv(path_to_result_dataframes_with_maverick + os.sep + "Evaluation" + os.sep + "Evaluation_F1Score", index_col=0, header=[0, 1])
    f1_score_without_maverick = pd.read_csv(path_to_result_dataframes_without_maverick + os.sep + "Evaluation" + os.sep + "Evaluation_F1Score", index_col=0, header=[0, 1])

    path_to_data = get_data_from_route(dataset_name, type_of_partition, additional_parameter, random_seed)

    labels_with_maverick = pd.DataFrame()
    labels_without_maverick = pd.DataFrame()

    number_of_clients = int(len(os.listdir(path_to_data)) / 4) - 1

    for client_number in range(number_of_clients):
        if client_number == maverick_number:
            labels_without_maverick = pd.concat([labels_without_maverick, pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_mav_data_excluded_y_test.csv", index_col=0).sum(axis=0)).T], ignore_index=True)
            labels_with_maverick = pd.concat([labels_with_maverick, pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_y_test.csv", index_col=0).sum(axis=0)).T], ignore_index=True)
        else:
            labels = pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_y_test.csv", index_col=0).sum(axis=0)).T
            labels_with_maverick = pd.concat([labels_with_maverick, labels], ignore_index=True)
            labels_without_maverick = pd.concat([labels_without_maverick, labels], ignore_index=True)

    labels_with_maverick_proportion = labels_with_maverick.sum(axis=0) / labels_with_maverick.to_numpy().sum()
    labels_without_maverick_proportion = labels_without_maverick.sum(axis=0) / labels_without_maverick.to_numpy().sum()

    f1_score_global_with_maverick = f1_score_with_maverick.loc[f1_score_with_maverick.index[-1], "Global"]
    f1_score_global_without_maverick = f1_score_without_maverick.loc[f1_score_without_maverick.index[-1], "Global"]

    # Aggregated statistics with raw values
    f1_score_difference_computed = f1_score_global_with_maverick - f1_score_global_without_maverick
    f1_score_difference_computed_without_mav_label: pd.DataFrame = f1_score_difference_computed.copy()
    f1_score_difference_computed_without_mav_label.iloc[maverick_label] = 0
    max_loss = f1_score_difference_computed_without_mav_label.min()
    avg_loss = f1_score_difference_computed_without_mav_label.mean()
    max_gain_beside_maverick = f1_score_difference_computed_without_mav_label.max()
    maverick_gain_label = f1_score_difference_computed.iloc[maverick_label]

    value_distance_dataframe["AggregationProcess"] = [aggregation_process]
    value_distance_dataframe["Dataset"] = [dataset_name]
    value_distance_dataframe["Partition"] = [additional_parameter]
    value_distance_dataframe["Metric"] = ["F1Score"]
    value_distance_dataframe["RandomSeed"] = [random_seed]
    value_distance_dataframe["MaverickOnlyLabelGain"] = [maverick_gain_label]
    value_distance_dataframe["MaverickMaxLoss"] = [max_loss]
    value_distance_dataframe["MaverickAvgLoss"] = [avg_loss]
    value_distance_dataframe["MaverickMaxGainBesidesOnlyLabel"] = [max_gain_beside_maverick]

    # Aggregated statistics with raw values
    f1_score_weighted_with_maverick = f1_score_global_with_maverick * labels_with_maverick_proportion
    f1_score_weighted_without_maverick = f1_score_global_without_maverick * labels_without_maverick_proportion
    f1_score_weighted_difference_computed = f1_score_weighted_with_maverick - f1_score_weighted_without_maverick
    f1_score_weighted_difference_computed_without_mav_label: pd.DataFrame = f1_score_weighted_difference_computed.copy()
    f1_score_weighted_difference_computed_without_mav_label.iloc[maverick_label] = 0
    max_loss = f1_score_weighted_difference_computed_without_mav_label.min()
    avg_loss = f1_score_weighted_difference_computed_without_mav_label.mean()
    max_gain_beside_maverick = f1_score_weighted_difference_computed_without_mav_label.max()
    maverick_gain_label = f1_score_weighted_difference_computed.iloc[maverick_label]

    value_distance_dataframe["MaverickWeightedOnlyLabelGain"] = [maverick_gain_label]
    value_distance_dataframe["MaverickWeightedMaxLoss"] = [max_loss]
    value_distance_dataframe["MaverickWeightedAvgLoss"] = [avg_loss]
    value_distance_dataframe["MaverickWeightedMaxGainBesidesOnlyLabel"] = [max_gain_beside_maverick]


    return value_distance_dataframe

In [115]:
f1_score_maverick_comparisons = pd.DataFrame()

training_configurations_mavericks = [scenario for scenario in training_configurations_for_distances
                                     if scenario["partition"] == "manual"
                                      and scenario.get("is_maverick_scenario", False) == True
                                      and scenario.get("exclude_maverick", False) == False
                                      and scenario.get("dataset") != "wids_dataset"]

for training_configuration in training_configurations_mavericks:
    aggregation_process = training_configuration["strategy"]
    dataset_name = training_configuration["dataset"]
    maverick_number = training_configuration["maverick_client_number"]
    maverick_label = training_configuration["maverick_special_class_or_columns_for_subpopulation"]
    type_of_partition = training_configuration["partition"]
    additional_parameter = training_configuration["name"]
    # additional_parameter = configuration["Dirichlet/NamePartition"]
    random_seed = training_configuration["run_number"]

    f1_score_maverick_comparisons = pd.concat([f1_score_maverick_comparisons, f1_score_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick_number, maverick_label)], ignore_index=True)

In [116]:
display(f1_score_maverick_comparisons)
f1_score_maverick_comparisons[f1_score_maverick_comparisons["Partition"] == "edgeiot_coreset_1_Maverick_Only_Normal"]

,AggregationProcess,Dataset,Partition,Metric,RandomSeed,MaverickOnlyLabelGain,MaverickMaxLoss,MaverickAvgLoss,MaverickMaxGainBesidesOnlyLabel,MaverickWeightedOnlyLabelGain,MaverickWeightedMaxLoss,MaverickWeightedAvgLoss,MaverickWeightedMaxGainBesidesOnlyLabel
0,FedAvg,har,HAR_1_Maverick_Laying,F1Score,1,0.980095,-0.013980,-0.003993,0.009470,0.179567,-0.038479,-0.029375,0.000000
1,FedAvg,har,HAR_1_Maverick_WalkingUpstairs,F1Score,1,0.867416,-0.067072,-0.025438,0.000000,0.139390,-0.036222,-0.029696,0.000000
2,FedAvg,har,HAR_1_Maverick_Laying_Balanced,F1Score,1,0.643478,-0.066739,-0.014753,0.004149,0.020766,-0.019148,-0.008031,0.000000
3,FedAvg,har,HAR_1_Maverick_WalkingUpstairs_Balanced,F1Score,1,0.477273,-0.021325,-0.002911,0.004184,0.012936,-0.009419,-0.004860,0.000000
4,FedAvg,har,HAR_1_Maverick_Laying,F1Score,2,0.982987,-0.017695,-0.004833,0.008357,0.180097,-0.040820,-0.029701,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_ddos_udp,F1Score,4,0.996769,-0.676407,-0.081311,0.040544,0.054032,-0.038256,-0.003929,0.000025
116,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Least_Class,F1Score,5,0.171429,-0.087860,-0.010478,0.032119,0.000038,-0.000769,-0.000121,0.000535
117,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,5,0.948029,-0.354272,-0.075476,0.000000,0.669797,-0.185805,-0.035443,0.000000
118,Scaffold,edge-iot-coreset,edgeiot_coreset_1_Maverick_sql_injection,F1Score,5,0.000000,-0.928571,-0.182480,0.000000,0.000000,-0.025713,-0.003526,0.000000


,AggregationProcess,Dataset,Partition,Metric,RandomSeed,MaverickOnlyLabelGain,MaverickMaxLoss,MaverickAvgLoss,MaverickMaxGainBesidesOnlyLabel,MaverickWeightedOnlyLabelGain,MaverickWeightedMaxLoss,MaverickWeightedAvgLoss,MaverickWeightedMaxGainBesidesOnlyLabel
21,FedAvg,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,1,0.945233,-0.699822,-0.150012,0.000064,0.667821,-0.185808,-0.036743,0.000000
25,FedAvg,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,2,0.946070,-0.322632,-0.062965,0.000045,0.668413,-0.185793,-0.035311,0.000000
29,FedAvg,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,3,0.947417,-0.263234,0.028465,0.548387,0.669364,-0.185730,-0.035331,0.000306
33,FedAvg,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,4,0.947591,-0.352060,-0.066015,0.202634,0.669487,-0.185815,-0.036195,0.000000
37,FedAvg,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,5,0.947050,-0.234797,0.030561,0.545455,0.669105,-0.185823,-0.034397,0.000305
61,FedProx,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,1,0.945025,-0.392027,-0.105383,0.000105,0.667674,-0.185855,-0.034171,0.000000
65,FedProx,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,2,0.938813,-0.557576,-0.122373,0.000000,0.663285,-0.185884,-0.034504,0.000000
69,FedProx,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,3,0.943950,-0.379283,-0.042120,0.181818,0.666915,-0.185064,-0.032447,0.000038
73,FedProx,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,4,0.945817,-0.663358,-0.123284,0.000039,0.668234,-0.185918,-0.034888,0.000000
77,FedProx,edge-iot-coreset,edgeiot_coreset_1_Maverick_Only_Normal,F1Score,5,0.942621,-0.703606,-0.145989,0.000000,0.665976,-0.185927,-0.034890,0.000000


In [117]:
display(f1_score_maverick_comparisons.groupby(["Dataset", "Partition"]).mean(numeric_only=True))
display(f1_score_maverick_comparisons.groupby(["Dataset", "Partition"]).mean(numeric_only=True).loc[:, ["MaverickOnlyLabelGain", "MaverickWeightedOnlyLabelGain", "MaverickWeightedAvgLoss"]])
display(f1_score_maverick_comparisons.groupby(["Dataset", "Partition"]).std(numeric_only=True).loc[:, ["MaverickOnlyLabelGain", "MaverickWeightedOnlyLabelGain", "MaverickWeightedAvgLoss"]])

RandomSeed  \
Dataset          Partition                                              
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class           3.0   
                 edgeiot_coreset_1_Maverick_Only_Normal           3.0   
                 edgeiot_coreset_1_Maverick_ddos_udp              3.0   
                 edgeiot_coreset_1_Maverick_sql_injection         3.0   
har              HAR_1_Maverick_Laying                            3.0   
                 HAR_1_Maverick_Laying_Balanced                   3.0   
                 HAR_1_Maverick_WalkingUpstairs                   3.0   
                 HAR_1_Maverick_WalkingUpstairs_Balanced          3.0   

                                                           MaverickOnlyLabelGain  \
Dataset          Partition                                                         
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                 0.475545   
                 edgeiot_coreset_1_Maverick_Only_Normal                 0.945453   
                 edgeiot_coreset_1_Maverick_ddos_udp                    0.960821   
                 edgeiot_coreset_1_Maverick_sql_injection               0.000000   
har              HAR_1_Maverick_Laying                                  0.992159   
                 HAR_1_Maverick_Laying_Balanced                         0.492989   
                 HAR_1_Maverick_WalkingUpstairs                         0.885692   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                0.290634   

                                                           MaverickMaxLoss  \
Dataset          Partition                                                   
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class          -0.060564   
                 edgeiot_coreset_1_Maverick_Only_Normal          -0.472194   
                 edgeiot_coreset_1_Maverick_ddos_udp             -0.253779   
                 edgeiot_coreset_1_Maverick_sql_injection        -0.482060   
har              HAR_1_Maverick_Laying                           -0.009589   
                 HAR_1_Maverick_Laying_Balanced                  -0.043025   
                 HAR_1_Maverick_WalkingUpstairs                  -0.044285   
                 HAR_1_Maverick_WalkingUpstairs_Balanced         -0.023010   

                                                           MaverickAvgLoss  \
Dataset          Partition                                                   
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class           0.008091   
                 edgeiot_coreset_1_Maverick_Only_Normal          -0.081981   
                 edgeiot_coreset_1_Maverick_ddos_udp             -0.026325   
                 edgeiot_coreset_1_Maverick_sql_injection        -0.099740   
har              HAR_1_Maverick_Laying                            0.000379   
                 HAR_1_Maverick_Laying_Balanced                  -0.009641   
                 HAR_1_Maverick_WalkingUpstairs                  -0.013977   
                 HAR_1_Maverick_WalkingUpstairs_Balanced         -0.006246   

                                                           MaverickMaxGainBesidesOnlyLabel  \
Dataset          Partition                                                                   
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                           0.137288   
                 edgeiot_coreset_1_Maverick_Only_Normal                           0.116772   
                 edgeiot_coreset_1_Maverick_ddos_udp                              0.089365   
                 edgeiot_coreset_1_Maverick_sql_injection                         0.020461   
har              HAR_1_Maverick_Laying                                            0.011240   
                 HAR_1_Maverick_Laying_Balanced                                   0.006878   
                 HAR_1_Maverick_WalkingUpstairs                                   0.006935   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                       

MaverickOnlyLabelGain  \
Dataset          Partition                                                         
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                 0.475545   
                 edgeiot_coreset_1_Maverick_Only_Normal                 0.945453   
                 edgeiot_coreset_1_Maverick_ddos_udp                    0.960821   
                 edgeiot_coreset_1_Maverick_sql_injection               0.000000   
har              HAR_1_Maverick_Laying                                  0.992159   
                 HAR_1_Maverick_Laying_Balanced                         0.492989   
                 HAR_1_Maverick_WalkingUpstairs                         0.885692   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                0.290634   

                                                           MaverickWeightedOnlyLabelGain  \
Dataset          Partition                                                                 
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                         0.000106   
                 edgeiot_coreset_1_Maverick_Only_Normal                         0.667976   
                 edgeiot_coreset_1_Maverick_ddos_udp                            0.052083   
                 edgeiot_coreset_1_Maverick_sql_injection                       0.000000   
har              HAR_1_Maverick_Laying                                          0.181777   
                 HAR_1_Maverick_Laying_Balanced                                 0.015909   
                 HAR_1_Maverick_WalkingUpstairs                                 0.142327   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                        0.007877   

                                                           MaverickWeightedAvgLoss  
Dataset          Partition                                                          
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                   0.000058  
                 edgeiot_coreset_1_Maverick_Only_Normal                  -0.035160  
                 edgeiot_coreset_1_Maverick_ddos_udp                     -0.003496  
                 edgeiot_coreset_1_Maverick_sql_injection                -0.002978  
har              HAR_1_Maverick_Laying                                   -0.028664  
                 HAR_1_Maverick_Laying_Balanced                          -0.006961  
                 HAR_1_Maverick_WalkingUpstairs                          -0.027690  
                 HAR_1_Maverick_WalkingUpstairs_Balanced                 -0.005451

MaverickOnlyLabelGain  \
Dataset          Partition                                                         
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                 0.404328   
                 edgeiot_coreset_1_Maverick_Only_Normal                 0.002330   
                 edgeiot_coreset_1_Maverick_ddos_udp                    0.077556   
                 edgeiot_coreset_1_Maverick_sql_injection               0.000000   
har              HAR_1_Maverick_Laying                                  0.005930   
                 HAR_1_Maverick_Laying_Balanced                         0.276643   
                 HAR_1_Maverick_WalkingUpstairs                         0.076145   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                0.200559   

                                                           MaverickWeightedOnlyLabelGain  \
Dataset          Partition                                                                 
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                         0.000090   
                 edgeiot_coreset_1_Maverick_Only_Normal                         0.001646   
                 edgeiot_coreset_1_Maverick_ddos_udp                            0.004204   
                 edgeiot_coreset_1_Maverick_sql_injection                       0.000000   
har              HAR_1_Maverick_Laying                                          0.001087   
                 HAR_1_Maverick_Laying_Balanced                                 0.008928   
                 HAR_1_Maverick_WalkingUpstairs                                 0.012236   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                        0.005436   

                                                           MaverickWeightedAvgLoss  
Dataset          Partition                                                          
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                   0.000207  
                 edgeiot_coreset_1_Maverick_Only_Normal                   0.001153  
                 edgeiot_coreset_1_Maverick_ddos_udp                      0.000727  
                 edgeiot_coreset_1_Maverick_sql_injection                 0.000936  
har              HAR_1_Maverick_Laying                                    0.000818  
                 HAR_1_Maverick_Laying_Balanced                           0.000920  
                 HAR_1_Maverick_WalkingUpstairs                           0.002797  
                 HAR_1_Maverick_WalkingUpstairs_Balanced                  0.000877

Checking the valuation per subpopulation

In [118]:
def mse_subpopulation_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick_number, maverick_categorical_features, maverick_value):
    mse_evaluation_difference = pd.DataFrame()

    path_to_result_dataframes_with_maverick = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True, False)
    path_to_result_dataframes_without_maverick = get_results_from_route(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, True, True)

    mse_with_maverick = pd.read_csv(path_to_result_dataframes_with_maverick + os.sep + "Evaluation" + os.sep + "Evaluation_Subpopulations.csv", index_col=[0, 1, 2, 3])
    mse_without_maverick = pd.read_csv(path_to_result_dataframes_without_maverick + os.sep + "Evaluation" + os.sep + "Evaluation_Subpopulations.csv", index_col=[0, 1, 2, 3])

    filtered_result_for_last_round_with_maverick = mse_with_maverick.loc[mse_with_maverick.index[-1][0], "Global", :, "MSE"].copy(deep=True)
    filtered_result_for_last_round_without_maverick = mse_without_maverick.loc[mse_without_maverick.index[-1][0], "Global", :, "MSE"].copy(deep=True)

    filtered_result_for_last_round_with_maverick.drop("Class", axis=1, inplace=True)
    filtered_result_for_last_round_without_maverick.drop("Class", axis=1, inplace=True)

    path_to_data = get_data_from_route(dataset_name, type_of_partition, additional_parameter, random_seed)

    categories_with_maverick = pd.DataFrame()
    categories_without_maverick = pd.DataFrame()

    number_of_clients = int(len(os.listdir(path_to_data)) / 4) - 1

    for client_number in range(number_of_clients):
        if client_number == maverick_number:
            categories_without_maverick = pd.concat([categories_without_maverick, pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_mav_data_excluded_X_test.csv", index_col=0).sum(axis=0)).T], ignore_index=True)
            categories_with_maverick = pd.concat([categories_with_maverick, pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_X_test.csv", index_col=0).sum(axis=0)).T], ignore_index=True)
        else:
            subpopulation_data = pd.DataFrame(pd.read_csv(path_to_data + os.sep + f"client_{str(client_number)}_X_test.csv", index_col=0).sum(axis=0)).T
            categories_with_maverick = pd.concat([categories_with_maverick, subpopulation_data], ignore_index=True)
            categories_without_maverick = pd.concat([categories_without_maverick, subpopulation_data], ignore_index=True)


    categories_with_maverick = categories_with_maverick.filter(regex="facility_type_.*", axis=1)
    categories_without_maverick = categories_without_maverick.filter(regex="facility_type_.*", axis=1)

    subpopulation_percentages_with_maverick = categories_with_maverick.sum(axis=0) / categories_with_maverick.to_numpy().sum()
    subpopulation_percentages_without_maverick = categories_without_maverick.sum(axis=0) / categories_without_maverick.to_numpy().sum()

    subpopulation_percentages_with_maverick.index = (
    subpopulation_percentages_with_maverick.index.str.replace('facility_type_', '', regex=False)
)
    subpopulation_percentages_without_maverick.index = (
    subpopulation_percentages_without_maverick.index.str.replace('facility_type_', '', regex=False)
)

    # Aggregated statistics with raw values
    mse_difference_computed = filtered_result_for_last_round_with_maverick - filtered_result_for_last_round_without_maverick
    mse_difference_computed_without_mav_label: pd.DataFrame = mse_difference_computed.copy()
    mse_difference_computed_without_mav_label.loc[maverick_value] = 0
    max_loss_gain = mse_difference_computed_without_mav_label.min().iat[0]
    avg_loss = mse_difference_computed_without_mav_label.mean().iat[0]
    max_loss_gain_beside_maverick = mse_difference_computed_without_mav_label.min().iat[0]
    max_impairment_by_maverick = mse_difference_computed_without_mav_label.max().iat[0]
    # Unsure why, but iat is needed to remove name
    maverick_gain_subpopulation = mse_difference_computed.loc[maverick_value].iat[0]

    mse_evaluation_difference["AggregationProcess"] = [aggregation_process]
    # mse_evaluation_difference["Dataset"] = [dataset_name]
    mse_evaluation_difference["Partition"] = [additional_parameter]
    # mse_evaluation_difference["Metric"] = ["MSE"]
    mse_evaluation_difference["RandomSeed"] = [random_seed]
    mse_evaluation_difference["MaverickOnlySubpopulationGain"] = [maverick_gain_subpopulation]
    mse_evaluation_difference["MaverickMaxLossGain"] = [max_loss_gain]
    mse_evaluation_difference["MaverickAvgLoss"] = [avg_loss]
    mse_evaluation_difference["MaverickMaxLossGainBesidesOnlyLabel"] = [max_loss_gain_beside_maverick]
    mse_evaluation_difference["MaverickMaxImpairment"] = [max_impairment_by_maverick]

    # display(filtered_result_for_last_round_with_maverick)
    # display(pd.Series(subpopulation_percentages_with_maverick, index=subpopulation_percentages_with_maverick.index))
    weighted_result_for_last_round_with_maverick = filtered_result_for_last_round_with_maverick.mul(subpopulation_percentages_with_maverick, axis=0)
    weighted_result_for_last_round_without_maverick = filtered_result_for_last_round_without_maverick.mul( subpopulation_percentages_without_maverick, axis=0)
    mse_weighted_difference_computed = weighted_result_for_last_round_with_maverick - weighted_result_for_last_round_without_maverick
    mse_weighted_difference_computed_without_mav_label: pd.DataFrame = mse_weighted_difference_computed.copy()
    mse_weighted_difference_computed_without_mav_label.loc[maverick_value] = 0
    max_loss_gain = mse_weighted_difference_computed_without_mav_label.min().iat[0]
    avg_loss = mse_weighted_difference_computed_without_mav_label.mean().iat[0]
    max_loss_gain_beside_maverick = mse_weighted_difference_computed_without_mav_label.min().iat[0]
    max_impairment_by_maverick = mse_weighted_difference_computed_without_mav_label.max().iat[0]
    # Unsure why, but iat is needed to remove name
    maverick_gain_subpopulation = mse_weighted_difference_computed.loc[maverick_value].iat[0]

    mse_evaluation_difference["MaverickWeightedOnlySubpopulationGain"] = [maverick_gain_subpopulation]
    mse_evaluation_difference["MaverickWeightedMaxLossGain"] = [max_loss_gain]
    mse_evaluation_difference["MaverickWeightedAvgLoss"] = [avg_loss]
    mse_evaluation_difference["MaverickWeightedMaxLossGainBesidesOnlyLabel"] = [max_loss_gain_beside_maverick]
    mse_evaluation_difference["MaverickWeightedMaxImpairment"] = [max_impairment_by_maverick]

    return mse_evaluation_difference

In [119]:
mse_subpopulation_maverick_comparisons= pd.DataFrame()

training_configurations_mavericks = [scenario for scenario in training_configurations_for_distances
                                     if scenario["partition"] == "manual"
                                      and scenario.get("is_maverick_scenario", False) == True
                                      and scenario.get("exclude_maverick", False) == False
                                      and scenario.get("dataset") == "wids_dataset"]

for training_configuration in training_configurations_mavericks:
    aggregation_process = training_configuration["strategy"]
    dataset_name = training_configuration["dataset"]
    type_of_partition = training_configuration["partition"]
    additional_parameter = training_configuration["name"]
    random_seed = training_configuration["run_number"]

    maverick_number = training_configuration["maverick_client_number"]
    maverick_columns_for_subpopulation = training_configuration["maverick_special_class_or_columns_for_subpopulation"]
    maverick_category_value = training_configuration["maverick_category_value"]

    mse_subpopulation_maverick_comparisons = pd.concat([mse_subpopulation_maverick_comparisons, mse_subpopulation_maverick(aggregation_process, dataset_name, type_of_partition, additional_parameter, random_seed, maverick_number, maverick_columns_for_subpopulation, maverick_category_value)], ignore_index=True)

In [120]:
# display(mse_subpopulation_maverick_comparisons)
display(mse_subpopulation_maverick_comparisons.groupby(["Partition"]).mean(numeric_only=True))
display(mse_subpopulation_maverick_comparisons.groupby(["Partition"]).mean(numeric_only=True).loc[:, ["MaverickOnlySubpopulationGain", "MaverickWeightedOnlySubpopulationGain", "MaverickWeightedAvgLoss"]])
display(mse_subpopulation_maverick_comparisons.groupby(["Partition"]).std(numeric_only=True).loc[:, ["MaverickOnlySubpopulationGain", "MaverickWeightedOnlySubpopulationGain", "MaverickWeightedAvgLoss"]])
display(mse_subpopulation_maverick_comparisons.groupby(["AggregationProcess", "Partition"]).mean(numeric_only=True))
display(mse_subpopulation_maverick_comparisons.groupby(["AggregationProcess", "Partition"]).std(numeric_only=True))

display(mse_subpopulation_maverick_comparisons.groupby(["AggregationProcess", "Partition"]).mean(numeric_only=True).loc[:, ["MaverickOnlySubpopulationGain", "MaverickWeightedOnlySubpopulationGain", "MaverickWeightedAvgLoss"]])
display(mse_subpopulation_maverick_comparisons.groupby(["AggregationProcess", "Partition"]).std(numeric_only=True).loc[:, ["MaverickOnlySubpopulationGain", "MaverickWeightedOnlySubpopulationGain", "MaverickWeightedAvgLoss"]])

,RandomSeed,MaverickOnlySubpopulationGain,MaverickMaxLossGain,MaverickAvgLoss,MaverickMaxLossGainBesidesOnlyLabel,MaverickMaxImpairment,MaverickWeightedOnlySubpopulationGain,MaverickWeightedMaxLossGain,MaverickWeightedAvgLoss,MaverickWeightedMaxLossGainBesidesOnlyLabel,MaverickWeightedMaxImpairment
Partition,,,,,,,,,,,
wids_energy_maverick_facility_type_grocery_store,3.0,-95.694922,-1315.219043,9.495405,-1315.219043,1738.659523,3.291384,-5.139484,-0.118825,-5.139484,3.905827
wids_energy_maverick_facility_type_uncategorized_multifamily,3.0,-28.995095,-5086.026451,-387.578343,-5086.026451,1347.406757,31.333824,-30.807772,-3.644907,-30.807772,0.262407


,MaverickOnlySubpopulationGain,MaverickWeightedOnlySubpopulationGain,MaverickWeightedAvgLoss
Partition,,,
wids_energy_maverick_facility_type_grocery_store,-95.694922,3.291384,-0.118825
wids_energy_maverick_facility_type_uncategorized_multifamily,-28.995095,31.333824,-3.644907


,MaverickOnlySubpopulationGain,MaverickWeightedOnlySubpopulationGain,MaverickWeightedAvgLoss
Partition,,,
wids_energy_maverick_facility_type_grocery_store,640.650567,2.285634,0.566224
wids_energy_maverick_facility_type_uncategorized_multifamily,171.963834,81.351801,7.561519


RandomSeed  \
AggregationProcess Partition                                                        
FedAvg             wids_energy_maverick_facility_type_grocery_store           3.0   
                   wids_energy_maverick_facility_type_uncategorize...         3.0   
FedProx            wids_energy_maverick_facility_type_grocery_store           3.0   
                   wids_energy_maverick_facility_type_uncategorize...         3.0   
Scaffold           wids_energy_maverick_facility_type_grocery_store           3.0   
                   wids_energy_maverick_facility_type_uncategorize...         3.0   

                                                                       MaverickOnlySubpopulationGain  \
AggregationProcess Partition                                                                           
FedAvg             wids_energy_maverick_facility_type_grocery_store                        49.173326   
                   wids_energy_maverick_facility_type_uncategorize...                    -112.016350   
FedProx            wids_energy_maverick_facility_type_grocery_store                      -219.212637   
                   wids_energy_maverick_facility_type_uncategorize...                      14.531050   
Scaffold           wids_energy_maverick_facility_type_grocery_store                      -117.045454   
                   wids_energy_maverick_facility_type_uncategorize...                      10.500014   

                                                                       MaverickMaxLossGain  \
AggregationProcess Partition                                                                 
FedAvg             wids_energy_maverick_facility_type_grocery_store           -1138.571606   
                   wids_energy_maverick_facility_type_uncategorize...        -12862.352079   
FedProx            wids_energy_maverick_facility_type_grocery_store           -1259.616328   
                   wids_energy_maverick_facility_type_uncategorize...         -1151.121181   
Scaffold           wids_energy_maverick_facility_type_grocery_store           -1547.469196   
                   wids_energy_maverick_facility_type_uncategorize...         -1244.606092   

                                                                       MaverickAvgLoss  \
AggregationProcess Partition                                                             
FedAvg             wids_energy_maverick_facility_type_grocery_store           1.320062   
                   wids_energy_maverick_facility_type_uncategorize...     -1212.933908   
FedProx            wids_energy_maverick_facility_type_grocery_store          40.567021   
                   wids_energy_maverick_facility_type_uncategorize...        39.611555   
Scaffold           wids_energy_maverick_facility_type_grocery_store         -13.400866   
                   wids_energy_maverick_facility_type_uncategorize...        10.587325   

                                                                       MaverickMaxLossGainBesidesOnlyLabel  \
AggregationProcess Partition                                                                                 
FedAvg             wids_energy_maverick_facility_type_grocery_store                           -1138.571606   
                   wids_energy_maverick_facility_type_uncategorize...                        -12862.352079   
FedProx            wids_energy_maverick_facility_type_grocery_store                           -1259.616328   
                   wids_energy_maverick_facility_type_uncategorize...                         -1151.121181   
Scaffold           wids_energy_maverick_facility_type_grocery_store                           -1547.469196   
                   wids_energy_maverick_facility_type_uncategorize...                         -1244.606092   

                                                                       MaverickMaxImpairment  \
AggregationProcess Partition                                                                   
FedAvg    

RandomSeed  \
AggregationProcess Partition                                                        
FedAvg             wids_energy_maverick_facility_type_grocery_store      1.581139   
                   wids_energy_maverick_facility_type_uncategorize...    1.581139   
FedProx            wids_energy_maverick_facility_type_grocery_store      1.581139   
                   wids_energy_maverick_facility_type_uncategorize...    1.581139   
Scaffold           wids_energy_maverick_facility_type_grocery_store      1.581139   
                   wids_energy_maverick_facility_type_uncategorize...    1.581139   

                                                                       MaverickOnlySubpopulationGain  \
AggregationProcess Partition                                                                           
FedAvg             wids_energy_maverick_facility_type_grocery_store                       603.050480   
                   wids_energy_maverick_facility_type_uncategorize...                     285.114878   
FedProx            wids_energy_maverick_facility_type_grocery_store                       641.017678   
                   wids_energy_maverick_facility_type_uncategorize...                      64.890362   
Scaffold           wids_energy_maverick_facility_type_grocery_store                       784.900233   
                   wids_energy_maverick_facility_type_uncategorize...                      71.172753   

                                                                       MaverickMaxLossGain  \
AggregationProcess Partition                                                                 
FedAvg             wids_energy_maverick_facility_type_grocery_store             382.094113   
                   wids_energy_maverick_facility_type_uncategorize...         25538.606471   
FedProx            wids_energy_maverick_facility_type_grocery_store             833.741181   
                   wids_energy_maverick_facility_type_uncategorize...           381.224362   
Scaffold           wids_energy_maverick_facility_type_grocery_store            2249.426914   
                   wids_energy_maverick_facility_type_uncategorize...           653.246634   

                                                                       MaverickAvgLoss  \
AggregationProcess Partition                                                             
FedAvg             wids_energy_maverick_facility_type_grocery_store          61.035468   
                   wids_energy_maverick_facility_type_uncategorize...      2692.344186   
FedProx            wids_energy_maverick_facility_type_grocery_store          68.980381   
                   wids_energy_maverick_facility_type_uncategorize...        84.080192   
Scaffold           wids_energy_maverick_facility_type_grocery_store         100.196657   
                   wids_energy_maverick_facility_type_uncategorize...        46.183952   

                                                                       MaverickMaxLossGainBesidesOnlyLabel  \
AggregationProcess Partition                                                                                 
FedAvg             wids_energy_maverick_facility_type_grocery_store                             382.094113   
                   wids_energy_maverick_facility_type_uncategorize...                         25538.606471   
FedProx            wids_energy_maverick_facility_type_grocery_store                             833.741181   
                   wids_energy_maverick_facility_type_uncategorize...                           381.224362   
Scaffold           wids_energy_maverick_facility_type_grocery_store                            2249.426914   
                   wids_energy_maverick_facility_type_uncategorize...                           653.246634   

                                                                       MaverickMaxImpairment  \
AggregationProcess Partition                                                                   
FedAvg    

MaverickOnlySubpopulationGain  \
AggregationProcess Partition                                                                           
FedAvg             wids_energy_maverick_facility_type_grocery_store                        49.173326   
                   wids_energy_maverick_facility_type_uncategorize...                    -112.016350   
FedProx            wids_energy_maverick_facility_type_grocery_store                      -219.212637   
                   wids_energy_maverick_facility_type_uncategorize...                      14.531050   
Scaffold           wids_energy_maverick_facility_type_grocery_store                      -117.045454   
                   wids_energy_maverick_facility_type_uncategorize...                      10.500014   

                                                                       MaverickWeightedOnlySubpopulationGain  \
AggregationProcess Partition                                                                                   
FedAvg             wids_energy_maverick_facility_type_grocery_store                                 3.477163   
                   wids_energy_maverick_facility_type_uncategorize...                              -7.660172   
FedProx            wids_energy_maverick_facility_type_grocery_store                                 3.185287   
                   wids_energy_maverick_facility_type_uncategorize...                              51.859396   
Scaffold           wids_energy_maverick_facility_type_grocery_store                                 3.211701   
                   wids_energy_maverick_facility_type_uncategorize...                              49.802246   

                                                                       MaverickWeightedAvgLoss  
AggregationProcess Partition                                                                    
FedAvg             wids_energy_maverick_facility_type_grocery_store                   0.082347  
                   wids_energy_maverick_facility_type_uncategorize...                -7.531293  
FedProx            wids_energy_maverick_facility_type_grocery_store                  -0.033967  
                   wids_energy_maverick_facility_type_uncategorize...                -1.740538  
Scaffold           wids_energy_maverick_facility_type_grocery_store                  -0.404856  
                   wids_energy_maverick_facility_type_uncategorize...                -1.662890

MaverickOnlySubpopulationGain  \
AggregationProcess Partition                                                                           
FedAvg             wids_energy_maverick_facility_type_grocery_store                       603.050480   
                   wids_energy_maverick_facility_type_uncategorize...                     285.114878   
FedProx            wids_energy_maverick_facility_type_grocery_store                       641.017678   
                   wids_energy_maverick_facility_type_uncategorize...                      64.890362   
Scaffold           wids_energy_maverick_facility_type_grocery_store                       784.900233   
                   wids_energy_maverick_facility_type_uncategorize...                      71.172753   

                                                                       MaverickWeightedOnlySubpopulationGain  \
AggregationProcess Partition                                                                                   
FedAvg             wids_energy_maverick_facility_type_grocery_store                                 2.454069   
                   wids_energy_maverick_facility_type_uncategorize...                             134.981178   
FedProx            wids_energy_maverick_facility_type_grocery_store                                 2.412899   
                   wids_energy_maverick_facility_type_uncategorize...                              30.536815   
Scaffold           wids_energy_maverick_facility_type_grocery_store                                 2.524826   
                   wids_energy_maverick_facility_type_uncategorize...                              34.019803   

                                                                       MaverickWeightedAvgLoss  
AggregationProcess Partition                                                                    
FedAvg             wids_energy_maverick_facility_type_grocery_store                   0.183709  
                   wids_energy_maverick_facility_type_uncategorize...                13.104766  
FedProx            wids_energy_maverick_facility_type_grocery_store                   0.233815  
                   wids_energy_maverick_facility_type_uncategorize...                 0.189408  
Scaffold           wids_energy_maverick_facility_type_grocery_store                   0.933734  
                   wids_energy_maverick_facility_type_uncategorize...                 0.153205

We mark as unreliable those trainings that have a bigger standard deviation than the mean value. They will be classified as harmless.

In [156]:
grouped_f1_score_maverick_comparisons = f1_score_maverick_comparisons.groupby(["Dataset", "Partition"]).mean(numeric_only=True).loc[:, ["MaverickOnlyLabelGain", "MaverickWeightedOnlyLabelGain", "MaverickWeightedAvgLoss"]]
grouped_mse_subpopulation_maverick_comparisons = mse_subpopulation_maverick_comparisons.groupby(["AggregationProcess", "Partition"]).mean(numeric_only=True).loc[:, ["MaverickOnlySubpopulationGain", "MaverickWeightedOnlySubpopulationGain", "MaverickWeightedAvgLoss"]]
grouped_f1_score_maverick_comparisons_std = f1_score_maverick_comparisons.groupby(["Dataset", "Partition"]).std(numeric_only=True).loc[:, ["MaverickOnlyLabelGain", "MaverickWeightedOnlyLabelGain", "MaverickWeightedAvgLoss"]]
grouped_mse_subpopulation_maverick_comparisons_std = mse_subpopulation_maverick_comparisons.groupby(["AggregationProcess", "Partition"]).std(numeric_only=True).loc[:, ["MaverickOnlySubpopulationGain", "MaverickWeightedOnlySubpopulationGain", "MaverickWeightedAvgLoss"]]

grouped_f1_score_maverick_comparisons["Reliable"] = grouped_f1_score_maverick_comparisons["MaverickOnlyLabelGain"] > grouped_f1_score_maverick_comparisons_std["MaverickOnlyLabelGain"]
grouped_mse_subpopulation_maverick_comparisons["Reliable"] = grouped_mse_subpopulation_maverick_comparisons["MaverickOnlySubpopulationGain"] > grouped_mse_subpopulation_maverick_comparisons_std["MaverickOnlySubpopulationGain"]


In [157]:
grouped_f1_score_maverick_comparisons

MaverickOnlyLabelGain  \
Dataset          Partition                                                         
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                 0.475545   
                 edgeiot_coreset_1_Maverick_Only_Normal                 0.945453   
                 edgeiot_coreset_1_Maverick_ddos_udp                    0.960821   
                 edgeiot_coreset_1_Maverick_sql_injection               0.000000   
har              HAR_1_Maverick_Laying                                  0.992159   
                 HAR_1_Maverick_Laying_Balanced                         0.492989   
                 HAR_1_Maverick_WalkingUpstairs                         0.885692   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                0.290634   

                                                           MaverickWeightedOnlyLabelGain  \
Dataset          Partition                                                                 
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                         0.000106   
                 edgeiot_coreset_1_Maverick_Only_Normal                         0.667976   
                 edgeiot_coreset_1_Maverick_ddos_udp                            0.052083   
                 edgeiot_coreset_1_Maverick_sql_injection                       0.000000   
har              HAR_1_Maverick_Laying                                          0.181777   
                 HAR_1_Maverick_Laying_Balanced                                 0.015909   
                 HAR_1_Maverick_WalkingUpstairs                                 0.142327   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                        0.007877   

                                                           MaverickWeightedAvgLoss  \
Dataset          Partition                                                           
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                   0.000058   
                 edgeiot_coreset_1_Maverick_Only_Normal                  -0.035160   
                 edgeiot_coreset_1_Maverick_ddos_udp                     -0.003496   
                 edgeiot_coreset_1_Maverick_sql_injection                -0.002978   
har              HAR_1_Maverick_Laying                                   -0.028664   
                 HAR_1_Maverick_Laying_Balanced                          -0.006961   
                 HAR_1_Maverick_WalkingUpstairs                          -0.027690   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                 -0.005451   

                                                           Reliable  
Dataset          Partition                                           
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class        True  
                 edgeiot_coreset_1_Maverick_Only_Normal        True  
                 edgeiot_coreset_1_Maverick_ddos_udp           True  
                 edgeiot_coreset_1_Maverick_sql_injection     False  
har              HAR_1_Maverick_Laying                         True  
                 HAR_1_Maverick_Laying_Balanced                True  
                 HAR_1_Maverick_WalkingUpstairs                True  
                 HAR_1_Maverick_WalkingUpstairs_Balanced       True

In [158]:
grouped_mse_subpopulation_maverick_comparisons

MaverickOnlySubpopulationGain  \
AggregationProcess Partition                                                                           
FedAvg             wids_energy_maverick_facility_type_grocery_store                        49.173326   
                   wids_energy_maverick_facility_type_uncategorize...                    -112.016350   
FedProx            wids_energy_maverick_facility_type_grocery_store                      -219.212637   
                   wids_energy_maverick_facility_type_uncategorize...                      14.531050   
Scaffold           wids_energy_maverick_facility_type_grocery_store                      -117.045454   
                   wids_energy_maverick_facility_type_uncategorize...                      10.500014   

                                                                       MaverickWeightedOnlySubpopulationGain  \
AggregationProcess Partition                                                                                   
FedAvg             wids_energy_maverick_facility_type_grocery_store                                 3.477163   
                   wids_energy_maverick_facility_type_uncategorize...                              -7.660172   
FedProx            wids_energy_maverick_facility_type_grocery_store                                 3.185287   
                   wids_energy_maverick_facility_type_uncategorize...                              51.859396   
Scaffold           wids_energy_maverick_facility_type_grocery_store                                 3.211701   
                   wids_energy_maverick_facility_type_uncategorize...                              49.802246   

                                                                       MaverickWeightedAvgLoss  \
AggregationProcess Partition                                                                     
FedAvg             wids_energy_maverick_facility_type_grocery_store                   0.082347   
                   wids_energy_maverick_facility_type_uncategorize...                -7.531293   
FedProx            wids_energy_maverick_facility_type_grocery_store                  -0.033967   
                   wids_energy_maverick_facility_type_uncategorize...                -1.740538   
Scaffold           wids_energy_maverick_facility_type_grocery_store                  -0.404856   
                   wids_energy_maverick_facility_type_uncategorize...                -1.662890   

                                                                       Reliable  
AggregationProcess Partition                                                     
FedAvg             wids_energy_maverick_facility_type_grocery_store       False  
                   wids_energy_maverick_facility_type_uncategorize...     False  
FedProx            wids_energy_maverick_facility_type_grocery_store       False  
                   wids_energy_maverick_facility_type_uncategorize...     False  
Scaffold           wids_energy_maverick_facility_type_grocery_store       False  
                   wids_energy_maverick_facility_type_uncategorize...     False

In [ ]:
# def performance_drop_classification(performance_drop_aggregated_dataset):
#     """
#     We use an or for the performance drop across aggregation processed, due to lack of space in the paper.
#     Returns:
#
#     """
#     performance_drop_aggregated_dataset.loc[""]
#
# def value_harm_harmless(performance_drop_aggregated_dataset, grouped_dataset_mean, grouped_dataset_std):
#     grouped_dataset["Value"] =

In [159]:
# grouped_f1_score_maverick_comparisons["PerformanceDrop"] = pd.NA
# grouped_mse_subpopulation_maverick_comparisons["PerformanceDrop"] = pd.NA

grouped_f1_score_maverick_comparisons["Value"] = pd.NA
grouped_mse_subpopulation_maverick_comparisons["Value"] = pd.NA

strategies = ["FedAvg", "FedProx", "Scaffold"]
manual_partition_names = {
    "har": ["HAR_1_Maverick_Laying", "HAR_1_Maverick_WalkingUpstairs", "HAR_1_Maverick_Laying_Balanced", "HAR_1_Maverick_WalkingUpstairs_Balanced"],
    "edge-iot-coreset": ["edgeiot_coreset_1_Maverick_Least_Class", "edgeiot_coreset_1_Maverick_Only_Normal", "edgeiot_coreset_1_Maverick_sql_injection", "edgeiot_coreset_1_Maverick_ddos_udp"],
    "wids_dataset": ["wids_energy_maverick_facility_type_grocery_store", "wids_energy_maverick_facility_type_uncategorized_multifamily"]
}

for dataset, partitions in manual_partition_names.items():
    for partition in partitions:
        if dataset != "wids_dataset":
            performance_drop_for_classification = any([performance_dropped_results_aggregated.loc[(strategy, dataset, "manual", partition), "PerformanceDrop"] for strategy in strategies])
            if performance_drop_for_classification:
                grouped_f1_score_maverick_comparisons.loc[(dataset, partition), "Value"] = "Harm"
            else:
                if not grouped_f1_score_maverick_comparisons.loc[(dataset, partition), "Reliable"]:
                    grouped_f1_score_maverick_comparisons.loc[(dataset, partition), "Value"] = "Harmless"
                else:
                    grouped_f1_score_maverick_comparisons.loc[(dataset, partition), "Value"] = "Value" if grouped_f1_score_maverick_comparisons.loc[(dataset, partition), "MaverickWeightedOnlyLabelGain"] >  grouped_f1_score_maverick_comparisons.loc[(dataset, partition), "MaverickWeightedAvgLoss"] else "Harmless"
        else:
            for strategy in strategies:
                performance_drop_for_regression = performance_dropped_results_aggregated.loc[(strategy, dataset, "manual", partition), "PerformanceDrop"]
                if performance_drop_for_regression:
                    grouped_mse_subpopulation_maverick_comparisons.loc[(strategy, partition), "Value"] = "Harm"
                else:
                    if not grouped_mse_subpopulation_maverick_comparisons.loc[(strategy, partition), "Reliable"]:
                        grouped_mse_subpopulation_maverick_comparisons.loc[(strategy, partition), "Value"] = "Harmless"
                    else:
                        grouped_mse_subpopulation_maverick_comparisons.loc[(strategy, partition), "Value"] = "Value" if grouped_mse_subpopulation_maverick_comparisons.loc[(strategy, partition), "MaverickWeightedOnlySubpopulationGain"] >  grouped_mse_subpopulation_maverick_comparisons.loc[(strategy, partition), "MaverickWeightedAvgLoss"] else "Harmless"


There is an error with Only Normal for this version, as the original value was taking from a wrong subset of examples. It will be corrected for the next version of the paper.

In [160]:
display(grouped_f1_score_maverick_comparisons)
display(grouped_mse_subpopulation_maverick_comparisons)

MaverickOnlyLabelGain  \
Dataset          Partition                                                         
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                 0.475545   
                 edgeiot_coreset_1_Maverick_Only_Normal                 0.945453   
                 edgeiot_coreset_1_Maverick_ddos_udp                    0.960821   
                 edgeiot_coreset_1_Maverick_sql_injection               0.000000   
har              HAR_1_Maverick_Laying                                  0.992159   
                 HAR_1_Maverick_Laying_Balanced                         0.492989   
                 HAR_1_Maverick_WalkingUpstairs                         0.885692   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                0.290634   

                                                           MaverickWeightedOnlyLabelGain  \
Dataset          Partition                                                                 
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                         0.000106   
                 edgeiot_coreset_1_Maverick_Only_Normal                         0.667976   
                 edgeiot_coreset_1_Maverick_ddos_udp                            0.052083   
                 edgeiot_coreset_1_Maverick_sql_injection                       0.000000   
har              HAR_1_Maverick_Laying                                          0.181777   
                 HAR_1_Maverick_Laying_Balanced                                 0.015909   
                 HAR_1_Maverick_WalkingUpstairs                                 0.142327   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                        0.007877   

                                                           MaverickWeightedAvgLoss  \
Dataset          Partition                                                           
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class                   0.000058   
                 edgeiot_coreset_1_Maverick_Only_Normal                  -0.035160   
                 edgeiot_coreset_1_Maverick_ddos_udp                     -0.003496   
                 edgeiot_coreset_1_Maverick_sql_injection                -0.002978   
har              HAR_1_Maverick_Laying                                   -0.028664   
                 HAR_1_Maverick_Laying_Balanced                          -0.006961   
                 HAR_1_Maverick_WalkingUpstairs                          -0.027690   
                 HAR_1_Maverick_WalkingUpstairs_Balanced                 -0.005451   

                                                           Reliable  Value  
Dataset          Partition                                                  
edge-iot-coreset edgeiot_coreset_1_Maverick_Least_Class        True  Value  
                 edgeiot_coreset_1_Maverick_Only_Normal        True  Value  
                 edgeiot_coreset_1_Maverick_ddos_udp           True  Value  
                 edgeiot_coreset_1_Maverick_sql_injection     False   Harm  
har              HAR_1_Maverick_Laying                         True  Value  
                 HAR_1_Maverick_Laying_Balanced                True  Value  
                 HAR_1_Maverick_WalkingUpstairs                True  Value  
                 HAR_1_Maverick_WalkingUpstairs_Balanced       True  Value

MaverickOnlySubpopulationGain  \
AggregationProcess Partition                                                                           
FedAvg             wids_energy_maverick_facility_type_grocery_store                        49.173326   
                   wids_energy_maverick_facility_type_uncategorize...                    -112.016350   
FedProx            wids_energy_maverick_facility_type_grocery_store                      -219.212637   
                   wids_energy_maverick_facility_type_uncategorize...                      14.531050   
Scaffold           wids_energy_maverick_facility_type_grocery_store                      -117.045454   
                   wids_energy_maverick_facility_type_uncategorize...                      10.500014   

                                                                       MaverickWeightedOnlySubpopulationGain  \
AggregationProcess Partition                                                                                   
FedAvg             wids_energy_maverick_facility_type_grocery_store                                 3.477163   
                   wids_energy_maverick_facility_type_uncategorize...                              -7.660172   
FedProx            wids_energy_maverick_facility_type_grocery_store                                 3.185287   
                   wids_energy_maverick_facility_type_uncategorize...                              51.859396   
Scaffold           wids_energy_maverick_facility_type_grocery_store                                 3.211701   
                   wids_energy_maverick_facility_type_uncategorize...                              49.802246   

                                                                       MaverickWeightedAvgLoss  \
AggregationProcess Partition                                                                     
FedAvg             wids_energy_maverick_facility_type_grocery_store                   0.082347   
                   wids_energy_maverick_facility_type_uncategorize...                -7.531293   
FedProx            wids_energy_maverick_facility_type_grocery_store                  -0.033967   
                   wids_energy_maverick_facility_type_uncategorize...                -1.740538   
Scaffold           wids_energy_maverick_facility_type_grocery_store                  -0.404856   
                   wids_energy_maverick_facility_type_uncategorize...                -1.662890   

                                                                       Reliable  \
AggregationProcess Partition                                                      
FedAvg             wids_energy_maverick_facility_type_grocery_store       False   
                   wids_energy_maverick_facility_type_uncategorize...     False   
FedProx            wids_energy_maverick_facility_type_grocery_store       False   
                   wids_energy_maverick_facility_type_uncategorize...     False   
Scaffold           wids_energy_maverick_facility_type_grocery_store       False   
                   wids_energy_maverick_facility_type_uncategorize...     False   

                                                                          Value  
AggregationProcess Partition                                                     
FedAvg             wids_energy_maverick_facility_type_grocery_store    Harmless  
                   wids_energy_maverick_facility_type_uncategorize...  Harmless  
FedProx            wids_energy_maverick_facility_type_grocery_store    Harmless  
                   wids_energy_maverick_facility_type_uncategorize...  Harmless  
Scaffold           wids_energy_maverick_facility_type_grocery_store    Harmless  
                   wids_energy_maverick_facility_type_uncategorize...  Harmless